# LightGBM con futuro · diario de experimentos (Pablo y Juanma · `futuro/`)

> ⚠️ Usa información de **otros snapshots de la sesión** (pasado y futuro): rompe a propósito la regla 1 del equipo.
> Por eso vive en `futuro/` y guarda en `futuro/oof|preds|submissions`: el stacking normal de `main` no lo ve; solo
> `stacking/stack.py --futuro` lo combina (ver `futuro/README.md`).

**Qué es:** `futuro/m_lgb_future.py`, un LightGBM LambdaRank (parámetros de referencia de `EQUIPO.md`) que puntúa cada
tarea de un snapshot. Ninguna feature usa la etiqueta. Tiene **dos pasos**:

1. **Primer pase (LightGBM).** Bloques de features de `futuro/features_future.py` (opción `--feats`):

| Bloque | Archivo | Qué contiene |
|---|---|---|
| `base` | `features_future.py` | columnas originales del snapshot |
| `logic` | `features_future.py` | reglas dentro del snapshot: plazo por tipo, factibilidad, ventana típica, EDF… (`logic_*`) |
| `past` | `features_future.py` | cuánto lleva la tarea en la lista, cambios respecto al snapshot anterior, racha (`p_*`, `blk_*`, `gap_*`) |
| `future` | `features_future.py` | segundos hasta la siguiente instrucción al avión y su tipo, si encaja con la tarea, cuándo desaparece la tarea (`f_*`) |
| `clear` | `feat_clear.py` | segundos hasta que cambian nivel autorizado, rumbo, nivel de salida, zona, conflicto, ruta (`sx_*`) |
| `coord` | `feat_coord.py` | segundos hasta que cambia la coordinación de entrada/salida, assume, transfer (`sc_*`): **no aporta** |

2. **Segundo paso de sesión** (`--sesion`, `futuro/feat_session.py`, sección 9). Un logit condicional de 9 pesos sobre
   los scores del primer pase. Usa que **cada tarea se trabaja una sola vez en la sesión**: si ya fue la favorita en
   otra racha, aquí se baja. Encima se aplica el **suavizado temporal** `prob` dentro de la racha
   (`futuro/pruebas/suavizado.py`, secciones 7 y 9b).

**Vocabulario:** *snapshot* = un `sample_id` (una lista de tareas cada 3 s). *Racha* (`block`) = snapshots seguidos de
una sesión sin huecos de más de 4.5 s. *Tramo* = (racha, tarea correcta).

**Cómo usar este notebook**
1. Ejecuta *Preparación* una vez (calcula las features, ~1–2 min).
2. Cada bloque: **hipótesis** → **variantes** (UNA cosa cambia respecto a la base) → **tabla** → conclusión escrita.
3. Pruebas rápidas con 1 semilla: diferencias < ~0.005 son ruido. Lo que gane se confirma con 3 semillas.
4. Las pruebas largas se lanzan desde la terminal en gpu.dsic (varias a la vez, 8 hilos cada una) y aquí se apuntan
   sus cifras. Todo queda en `experiments/lgb_future_runs.csv` (`historial()` al final).

**Cómo leer las cifras:** `cv` = top-3 out-of-fold con `Data/folds.csv` (GroupKFold de 5 por controlador).
`min_atco` = peor controlador (importa porque el leaderboard privado es **un solo controlador**, el 023).
*CV anidada* = los parámetros de un posproceso se eligen con 4 folds y se miden en el 5.º (número honesto).

**Regla desde el 21/10 (sección 14): todo experimento se juzga con TODAS estas métricas, no solo con la media.**
`m_lgb_future.py` las imprime y las apunta en `experiments/lgb_future_runs.csv` (vía `futuro/metricas.py`), y
`futuro/pruebas/comparar.py <ref> <cand>` las da para cualquier par de OOF:

| Métrica | Columna del registro | Por qué |
|---|---|---|
| CV media y por fold | `cv`, `fold0`–`fold4` | la de siempre |
| peor controlador (y cuál) | `min_atco`, `worst_atco` | el privado es un solo controlador |
| mediana por sesión | `ses_median` | la media la mueven unas pocas sesiones (12d) |
| CV por escenario | `sc1`, `sc2`, `sc3` | el 023 vuela los tres; una mejora en uno solo no es robusta |
| frente a la referencia (`--ref`, por defecto `lgb_future_clear`: el 1.er pase con `<ref>`, el segundo paso con `<ref>_ses`) | `d_cv`, `d_ses_median`, `ses_up`/`ses_down`, `d_atco_mean`/`d_atco_median`, `atco_up`/`atco_down` | cuántas sesiones y controladores mejoran o empeoran |

Un cambio se adopta si mejora la media **y** la mediana por sesión, sube en más sesiones de las que baja y no hunde
al peor controlador. Ojo: las diferencias de ±0.2 entre versiones pueden ser ruido de semillas (14b).

**Qué no se repite de `LGB_adri.ipynb`** (solo se lee, no se toca): Adri ya probó, con datos limpios, sus grupos de
features (`snapshot`, `tipo`, `avion`, `tipo_estado`, `contexto`): sueltos aportan poco y todos juntos empeoran. Sus
candidatos de parámetros (`lento`, `regularizado`, `xendcg`) se probaron aquí con futuro (bloque 2).

# ESTADO ACTUAL Y CONTEXTO PARA RETOMAR (21/10, 02:30)

> Léelo entero antes de proponer nada. Resume las secciones 0–19 de este notebook. El detalle y las cifras de cada
> prueba están en su sección; aquí solo está lo necesario para no repetir trabajo.

## 1 · El reto y cómo trabajamos

- **Reto:** AWARE Prioritisation Challenge (Engage 2, Zagreb). Cada snapshot (`sample_id`) es la lista de tareas abiertas
  de un controlador aéreo; hay que dar los 3 `row_index` más probables. Métrica: **top-3 hit rate**.
- **Leaderboards:** el público es solo el controlador **020** (sesiones 020sc2 y 020sc3); el **privado, que decide, es
  el 023** (sesiones 023sc1–3). Kaggle cierra el **miércoles 21/10 a las 13:00 (hora de Zagreb)**. La nota final es la
  mejor de las 2 entregas seleccionadas.
- **Situación:** público **0.947**, ahora mismo **cuartos**. Los mejores tienen **0.957**. Para llegar al jurado hay
  que estar en el top 3.
- **Equipo:** Pablo (A), Adri (B), Juanma (C), Pau (D). **Solo Juanma sube a Kaggle.** No se toca el trabajo de Adri ni
  de Juanma sin pedirlo.
- **Máquina:** gpu.dsic.upv.es, 32 hilos. LightGBM en CPU: varias pruebas a la vez con `--threads 5–10`. Python:
  `/home/alumno.upv.es/ptorsim/venv-flow/bin/python`, siempre con ruta absoluta en `nohup`.
- **Git:** no hay credenciales de GitHub en gpu.dsic: se hacen commits pequeños y **Pablo hace el push**:
  `cd /home/alumno.upv.es/ptorsim/FlowDataLab && git pull --rebase --autostash && git push`.
  Los `futuro/*/stack_*` modificados son de Juanma: no commitearlos.
- **Cómo trabajar con Pablo:**
  1. Antes de lanzar nada, decirle qué se va a probar, para que el equipo no lo duplique.
  2. Tras cada commit, darle los comandos de push.
  3. Documentarlo todo aquí: hipótesis → variantes → tabla → conclusión.
  4. Sin redundancias y sin repetir pruebas ya hechas.
  5. No inventar cifras.

## 2 · Cómo se decide (regla de la sección 14)

- **Validación:** `Data/folds.csv` (GroupKFold de 5 por controlador). Las decisiones se toman por la CV, no por el público.
- **Métricas de cada experimento** (las calculan `futuro/metricas.py` y `futuro/pruebas/comparar.py <ref> <cand>`):
  - CV media;
  - **mediana por sesión**;
  - CV por escenario;
  - **peor controlador** (y cuál);
  - sesiones y controladores que suben o bajan frente a la referencia;
  - bootstrap por controladores.
- **Un cambio se adopta** si mejora la media **y** la mediana por sesión, sube en más sesiones de las que baja y no hunde
  al peor controlador.
- **Ruido:**
  - con 1 semilla, diferencias < 0.005 son ruido;
  - con 3 semillas, **±0.2 entre versiones es ruido de semillas** (mueve a todos los controladores a la vez y el
    bootstrap no lo ve);
  - lo que gane se confirma con 3 semillas y después con 10.
- **Posprocesos** (segundo paso, suavizado, umbrales): siempre con **CV anidada** (se elige en 4 folds y se mide en el 5.º).

## 3 · Lo que tenemos

```
features base + logic + past + future + clear [+ oráculo de coordinaciones]
   ─► LightGBM LambdaRank (parámetros de EQUIPO.md, task_index categórica), 10 semillas     futuro/m_lgb_future.py
   ─► segundo paso de sesión: logit condicional de 9 pesos (--sesion)                    futuro/feat_session.py
   ─► suavizado "prob" dentro de la racha, parámetros por CV anidada                     futuro/pruebas/suavizado.py
```

| Modelo (10 semillas + sesión + suavizado) | CV anidada | peor controlador | Kaggle público (020) | sección |
|---|---|---|---|---|
| `lgb_future` (sin `clear`, sin sesión, 3 semillas) | 0.901 | 0.769 | 0.942 | 0–6c |
| **sin oráculo:** `suav_lgb_future_clear10_ses` | 0.918 | 0.784 | **0.947** | 9b |
| oráculo completo: `suav_lgb_future_clear10_or_ses` | 0.923 | 0.769 | 0.940 | 12 |
| **oráculo de coordinaciones:** `suav_lgb_future_clear10_orcoord_ses` | **0.925** | **0.796** | 0.934 | 14d–e |
| Juanma: `suav_f5_clear_l63_10_ses` (63 hojas + `extra_trees`) | 0.920 | 0.778 | — | 13 |
| Juanma: stack con oráculo `stackf_or` | 0.926 | 0.781 | — | 15 |

- **Oráculo** (`futuro/feat_oracle.py`, `--oraculo --oraculo-var coord`): qué hicieron **otros** controladores del mismo
  escenario en el mismo momento (tiempo alineado). Se calcula por fold y nunca usa al propio controlador. Usa etiquetas
  de otras sesiones; el equipo lo aceptó como regla de juego.
- **CV frente a público:** el oráculo gana en CV (+0.7) y pierde en el público (−1.3). El público son ~915 snapshots
  (−1.3 ≈ 12 snapshots); según la CV, una caída así ocurre con probabilidad 0.08. El 020 es un controlador muy de
  coordinaciones y el oráculo cambia ~50 % de sus top-3 (14f). No hay fallo técnico.
- **Entregas finales recomendadas (cobertura: una con oráculo y otra sin él; nunca dos con oráculo):**
  1. `futuro/submissions/sub_suav_lgb_future_clear10_orcoord_ses.csv`: mejor CV con todos los criterios.
  2. `futuro/submissions/sub_suav_lgb_future_clear10_ses.csv`: sin oráculo, mejor en el público.

## 4 · Lo que hemos aprendido (por orden de importancia)

1. **Manda la información, no el modelo.** Sin `future` (siguiente instrucción al avión, cuándo desaparece la tarea) se
   cae ~5 puntos (19). XGBoost, CatBoost, la red, los stacks y los parámetros dan lo mismo o menos (5, 6c, 11e, 12a, 13).
2. **La estructura de las etiquetas es la segunda fuente** (9): cada tarea se trabaja en un solo episodio de la sesión.
   El segundo paso de sesión lo aprovecha (+0.7) y debe ser **pequeño** (un LightGBM en su lugar sobreajusta).
3. **`clear` suma** (+0.6 en el 1.er pase, −0.3 si se quita con sesión). `logic`, `past` y las variables de sesión no
   aportan ni restan (19). **Ningún bloque empeora el modelo.**
4. **Los errores son pocos episodios enteros** (19): 3655 tramos (racha con la misma respuesta), el 92.7 % acertados
   enteros. **198 tramos fallados enteros dan el 82 % de los fallos** (75 tramos, la mitad): coordinaciones 87,
   zone_conflict 27, return_to_route 19, exit_conditions 19.
5. **Techo de información:**
   - En los tramos acertados, la tarea etiquetada deja huella observable (instrucción compatible ≤ 60 s, desaparece
     ≤ 30 s o cambia su estado) el 88 % de las veces; en los fallados enteros, el 27 %.
   - **El 77 % de los tramos fallados no tiene huella ni futura ni pasada.** El experto marca algo con lo que el
     controlador no hace nada visible.
   - En las coordinaciones, el 42 % de los contextos casi idénticos de otro controlador tienen la etiqueta contraria (17).
6. **Entre controladores cambia la decisión, no lo observable** (17, 18):
   - un clasificador de dominio no distingue a un controlador fuera de muestra (AUC 0.55–0.69);
   - ponderar por estilo o por covariate shift no ayuda;
   - 023 es de estilo atípico (lento, listas más cortas): 023sc2 es más atípica que cualquier sesión etiquetada.
7. **Test se parece a train** (19): misma mezcla de tipos y misma cobertura de huella.

## 5 · Probado sin mejora (no repetir)

| Idea | Sección | Resultado |
|---|---|---|
| XGBoost / CatBoost / red con futuro / stacking | 5, 6c, 13 | 0.896 / 0.874 / 0.887 / ≤ 0.903 en el 1.er pase; no suman |
| bloque `coord` (`sc_*`), `block` (`bk_*`), `rel` (`rl_*`) | 6, 11a, 16 | ruido o peor tras el segundo paso |
| parámetros (hojas, `min_child`, `trunc`, árboles), `colsample`, `extra_trees` | 9b, 12a, 13 | ruido; solo 63 hojas + `extra_trees` juntos dan +0.2 (Juanma) |
| segundo paso: Sinkhorn, LightGBM, variantes de M/Pnext, gemelo, pesos por tipo | 9, 9b, 11b | ruido o peor |
| peso por racha/tramo; peso ×1.5/×2 a coordinaciones | 11e, 16 | ruido |
| especialista de coordinaciones | 16 | −0.09 |
| HMM, modelo por racha, etiquetas ruidosas, prior por controlador (EM), modelo por escenario (Juanma) | 15 | igual o peor |
| selector de modelos por escenario | 14g | −0.12 a −0.33 frente a `orcoord` solo |
| oráculo ponderado por estilo (T = 2, 1, 0.5) | 17 | +0.10 / −0.05 / −0.02, ruido |
| geometría (borde, tráfico que cruza el borde, niveles) | 18 | −0.02 a −0.39 (los datos no traen posiciones) |
| covariate shift (pesos de un clasificador de dominio) | 18 | −0.32 / −0.11; CV anidada −0.15 |
| quitar bloques / variables de sesión | 19 | nada resta; modelo ligero igual (0.9144) |
| beam search sobre la sesión | 18 | no se hace: forward-backward (HMM) es la versión exacta y no mejoró |

## 6 · Mapa del notebook

| Sección | Contenido |
|---|---|
| 0–4 | punto de partida, qué bloque aporta, parámetros, versión para el stacking |
| 5, 5b, 7 | errores frente a la red; suavizado temporal (Juanma, CV anidada) |
| 6, 6b, 6c | bloques `coord` y `clear`; XGBoost / CatBoost / stacking |
| **9, 9b** | **segundo paso de sesión** y suavizado encima |
| 10, 11 (a–e) | ideas tras el segundo paso: features por racha, variantes del segundo paso, pesos, mezclas |
| 12 (a–d) | parámetros con sesión; **oráculo**; Kaggle y ganancia por escenario |
| 13, 15 | tandas de Juanma (`extra_trees`, red con `clear`, stacks, HMM, prior, etiquetas ruidosas, modelo por racha) |
| **14 (a–g)** | **regla de decisión robusta; oráculo de coordinaciones (mejor modelo)**; Kaggle; diagnóstico del 020; selector |
| 16 | coordinaciones desde el modelo (peso, especialista, features relativas) |
| 17 | oráculo ponderado por estilo y auditoría de ambigüedad |
| 18 | geometría del tráfico y covariate shift |
| **19** | **ablación, análisis de errores por tramos (techo de información) y test frente a train** |
| Historial | todas las ejecuciones (`historial()`) |

## 7 · Mapa de archivos

| Archivo | Qué es |
|---|---|
| `futuro/m_lgb_future.py` | modelo principal. Opciones: `--feats`, `--drop`, `--sesion`, `--oraculo`, `--oraculo-var`, `--oraculo-temp`, `--peso*`, `--seeds`, `--threads`, `--ref`, `--name`, `--tag` |
| `futuro/features_future.py` | bloques `base`, `logic`, `future`, `past` y llamada a los demás (`feat_coord`, `feat_clear`, `feat_block`, `feat_rel`, `feat_geo`) |
| `futuro/feat_session.py` | segundo paso de sesión |
| `futuro/feat_oracle.py` | oráculo (variantes, ponderado por estilo, `estilo()`) |
| `futuro/metricas.py`, `futuro/pruebas/comparar.py` | métricas de la regla y comparación con bootstrap |
| `futuro/pruebas/suavizado.py` | suavizado (Juanma) |
| `futuro/pruebas/errores.py`, `auditoria_coord.py`, `covshift.py`, `especialista.py` | análisis de las secciones 16–19 |
| `futuro/pruebas/f5_*.py`, `f6_*.py`, `f7_tramo.py`, `resultados.jsonl` | pruebas de Juanma |
| `futuro/oof|preds|submissions/` | salidas (formato común); `futuro/pruebas/oof/` para los posprocesos |
| `experiments/lgb_future_runs.csv` | registro de todas las ejecuciones (`historial()` al final del notebook) |
| `logs/` | logs de las ejecuciones en segundo plano |

## 8 · Si se sigue: lo que queda por probar

1. **Elegir las 2 entregas pensando en 023.** Comparar en la CV con oráculo / sin oráculo y con / sin suavizado en los
   controladores parecidos a 023 (lentos, listas cortas). Barato y es lo que más puede mover el privado.
2. **Estudiar los 235 tramos acertados SIN huella futura** (19): el modelo acierta algunos episodios sin rastro, así que
   algo es predecible. Si se encuentra el patrón (p. ej. instrucción a 2–5 min, o la tarea más urgente por plazo),
   podría ayudar en los 153 fallados.
3. Para el jurado: «el 82 % de los errores son 198 episodios en los que el controlador no hace nada visible».


> ## Actualización de Juanma (21/10, 10:30): NOR y entregas finales (secciones 20–23)
>
> | Envío | CV (+ sesión) | peor controlador | CV 023 | Kaggle público |
> |---|---|---|---|---|
> | A `futuro/submissions/sub_suav_lgb_future_clear10_ses.csv` | 0.918 (anidada) | 0.784 | 91.04 | 0.947 |
> | B `futuro/submissions/sub_suav_lgb_future_clear10_orcoord_ses.csv` (oráculo de coord.) | 0.925 (anidada) | 0.796 | 91.22 | 0.934 |
> | **N** `futuro/pruebas/submissions/sub_f10_instr_l63_s10_ses.csv` (bloque `instr` + 63 hojas + `extra_trees`, sin oráculo) | 0.924 | 0.784 | 91.83 | **0.951** |
> | **NOR** `futuro/pruebas/submissions/sub_f10_instr_l63_or_s10_ses.csv` (N + oráculo de coord.) | **0.9278** | **0.802** | **91.94** | estimado 0.935–0.960 |
>
> **Entregas finales recomendadas: NOR + N** (una con oráculo y otra sin él; sección 23). Kaggle se queda con la mejor de las dos en el
> privado: si el oráculo funciona en el 023 cuenta NOR, si falla (lo que sugieren los dos envíos con oráculo en el público) cuenta N.
> Llevar dos con oráculo (B + NOR) gana solo ~0.2 en E[máx] por la CV y no cubre ese caso.

<!-- f11:cabecera_pablo -->
> ## Actualización de Pablo (21/10, 06:05): M30 = 0.7·C + 0.3·N, mejor modelo (sección 24)
>
> El organizador permite usar **toda la información de los datasets** (snapshots futuros incluidos; el oráculo usa etiquetas
> de train/validation). Envíos en `futuro/pruebas/submissions/`, validados (1863 snapshots, 3 `row_index` distintos y existentes):
>
> | Envío | CV | peor controlador | CV 023 | privado esperado (frente a N30) |
> |---|---|---|---|---|
> | N `sub_f10_instr_l63_s10_ses` (público 0.951) | 0.9242 | 0.784 | 0.9287 | — |
> | N30 `sub_f11_N30_ses` (N, 30 semillas) | 0.9234 | 0.781 | 0.9273 | — |
> | C30 `sub_f11_C30_ses` (N + oráculo de coordinaciones, 30 semillas) | 0.9283 | 0.793 | 0.9320 | +0.50 (P 0.75) |
> | **M30 `sub_f11_M30_30`** (0.7·C30 + 0.3·N30 en probabilidades; w por CV anidada) | **0.9290** | **0.793** | **0.9344** | **+0.56** (p10 −0.17; P 0.81) |
>
> **Pareja recomendada: M30 + N** (`futuro/pruebas/submissions/sub_f10_instr_l63_s10_ses.csv`, público 0.951): una con oráculo y
> otra sin él (sección 24l). El público castiga al oráculo en el 020 (H1: 5 de 5 envíos, r = 0.97; para M30 predice ~0.944),
> así que la CV puede sobrestimarlo en un controlador nuevo: con N queda cubierto. Si se confía del todo en la CV: M30 + B
> (`futuro/submissions/sub_suav_lgb_future_clear10_orcoord_ses.csv`), E[máx] simulado 93.08 frente a 92.92.
> La mezcla con N quita al oráculo su veto a coordinaciones raras (controlador 006: C 0.88, M30 0.91). Ninguna otra cosa
> de la sección 24 mejora a N: parámetros, 6 bloques de features nuevos, 5 variantes del oráculo, oráculo sin etiquetas,
> objetivo softmax, DART, XGBoost, relevancia graduada y segundos pasos nuevos (31 entrenamientos y ~12 pruebas sin
> reentrenar). Solo el oráculo.


## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "futuro") else Path.cwd()
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "cv")); sys.path.insert(0, str(ROOT / "futuro"))
import m_lgb_future

data = m_lgb_future.build_data()      # una sola vez; todas las pruebas lo reutilizan
REF = {"LGB limpio (Adri)": 0.648, "red con futuro": 0.887}

In [ ]:
# Configuración base = parámetros de referencia de EQUIPO.md con los 4 bloques de features. Cada experimento cambia UNA clave.
BASE = dict(feats="base,logic,past,future", n_estimators=400, lr=0.05, leaves=31, min_child=50, subsample=0.8,
            colsample=0.7, trunc=5, reg_lambda=0.0, objective="lambdarank")


def probar(nombre, motivo, semillas=1, guardar=False, **cambios):
    """Lanza la CV con BASE + cambios; queda apuntado en experiments/lgb_future_runs.csv con su motivo.
    guardar=True genera futuro/oof/, futuro/preds/ y futuro/submissions/ (solo para la versión que va al stacking)."""
    cfg = {**BASE, **cambios}
    args = ["--name", nombre, "--tag", motivo] + (["--quick"] if not guardar else [])
    for k, v in cfg.items():
        if isinstance(v, bool):                       # flags sin valor (sesion=True -> --sesion)
            args += [f"--{k}"] if v else []
        else:
            args += [f"--{k.replace('_', '-')}", str(v)]
    a = m_lgb_future.parse(args)
    a.seeds = semillas
    return m_lgb_future.run(a, data)


def lanzar(bloque, variantes, semillas=1, sobre=None):
    """variantes: lista de (nombre, motivo, {cambios}). Siempre incluye la base para comparar en igualdad."""
    sobre = sobre or {}
    res = [("base", "referencia", probar(f"{bloque}/base", "referencia", semillas, **sobre))]
    for nombre, motivo, cambios in variantes:
        res.append((nombre, motivo, probar(f"{bloque}/{nombre}", motivo, semillas, **{**sobre, **cambios})))
    return tabla_bloque(res)


def tabla_bloque(res):
    t = pd.DataFrame(res, columns=["variante", "por qué", "cv"])
    ref = t.loc[t.variante == "base", "cv"].iloc[0]
    t["Δ vs base"] = t.cv - ref
    t["veredicto"] = t["Δ vs base"].map(lambda x: "—" if abs(x) < 1e-9 else ("mejora" if x > 0.005 else
                                                   ("empeora" if x < -0.005 else "ruido")))
    return (t.sort_values("cv", ascending=False).style
            .format({"cv": "{:.4f}", "Δ vs base": "{:+.4f}"})
            .background_gradient(subset=["Δ vs base"], cmap="RdYlGn", vmin=-0.02, vmax=0.02).hide(axis="index"))


def historial(filtro=""):
    """Todo lo probado (también desde la terminal), ordenado por CV."""
    r = pd.read_csv(m_lgb_future.LOG)
    r = r[r.name.str.contains(filtro, regex=False)] if filtro else r
    cols = ["when", "name", "tag", "cv", "min_atco", "worst_atco", "ses_median", "sc1", "sc2", "sc3", "d_cv", "d_ses_median",
            "ses_up", "ses_down", "atco_up", "atco_down", "seeds", "feats", "n_feats", "leaves", "min_child", "lr",
            "n_estimators", "colsample", "secs"]
    return r[[c for c in cols if c in r]].sort_values("cv", ascending=False).reset_index(drop=True)

## 0 · Punto de partida

Base con 3 semillas, generada desde la terminal (`python futuro/m_lgb_future.py`): es la versión `lgb_future` que ya está en
`futuro/oof/` y `futuro/preds/` para Juanma. La celda la mide con la métrica del equipo (`cv/metric.py`).

In [ ]:
from metric import report
report(pd.read_csv(ROOT / "futuro" / "oof" / "lgb_future.csv"), "lgb_future (base, 3 semillas)")

## 1 · ¿Qué bloque de información aporta?

**Hipótesis:** casi todo el salto viene del bloque `future` (la siguiente instrucción delata la tarea); `past` y `logic` aportan poco.
Quitar un bloque cada vez dice cuánto vale cada uno. Si quitar uno **no empeora**, sobra (menos ruido, más robustez).

| Variante | Qué se quita | Qué esperamos |
|---|---|---|
| sin `future` | f_* | caída grande (debería acercarse al LightGBM limpio + pasado) |
| sin `past` | p_*, blk_*, gap_* | caída pequeña |
| sin `logic` | logic_* | ruido: con futuro, las reglas a mano deberían sobrar |

In [ ]:
lanzar("feats", [
    ("sin_future", "¿cuánto aporta el futuro?",       dict(feats="base,logic,past")),
    ("sin_past",   "¿cuánto aporta el pasado?",        dict(feats="base,logic,future")),
    ("sin_logic",  "¿sobran las reglas a mano?",       dict(feats="base,past,future")),
])

**Conclusión bloque 1 · casi toda la información está en el futuro; pasado y reglas no aportan.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | |
|---|---|---|---|---|
| sin `logic` | 0.897 | −0.002 | 0.740 | ruido |
| sin `past` | 0.897 | −0.003 | 0.757 | ruido |
| sin `future` | **0.661** | **−0.239** | 0.596 | empeora muchísimo |

- **El bloque `future` es prácticamente todo el modelo:** sin él cae a 0.661, casi al nivel del LightGBM limpio de Adri (0.648).
- **`past` y `logic` no aportan con el futuro presente** (−0.002/−0.003, ruido): lo que dicen ya lo cuenta mejor la siguiente
  instrucción del controlador. No compensa quitarlos tampoco (no mejora), así que la base se queda con los 4 bloques.
- Sin futuro, `base+logic+past` da 0.661 frente a 0.648 de Adri: ~1 punto, que viene sobre todo del pasado (también rompe la
  regla 1). Las `logic_*` son solo del snapshot: Adri ya las tiene en `features.py` de `main` por si quiere probarlas.
- El peor controlador varía bastante entre variantes (0.740–0.757) con 1 semilla: no sacar conclusiones de ahí.

## 2 · Parámetros (los candidatos de Adri, sobre datos con futuro)

**Hipótesis:** con features tan informativas el modelo se apoya en pocas columnas muy fuertes (`f_next_input`…); lo que
le obligue a repartir (menos columnas por árbol) o a ser más prudente puede generalizar mejor a controladores nuevos.

| Variante | Cambios (iguales que en `LGB_adri.ipynb`) | Por qué |
|---|---|---|
| `lento` | `lr=0.03`, `n_estimators=700` | aprender más despacio con más árboles: suele generalizar mejor |
| `regularizado` | `min_child=100`, `colsample=0.5`, `reg_lambda=5` | hojas más grandes, menos columnas por árbol, pesos más pequeños |
| `xendcg` | `objective="rank_xendcg"` | otra pérdida de ranking, más estable con etiquetas ruidosas |

~1 min por variante (1 semilla).

In [ ]:
lanzar("params", [
    ("lento",        "más lento con más árboles (Adri)",                 dict(lr=0.03, n_estimators=700)),
    ("regularizado", "hojas grandes, menos columnas, L2 (Adri)",         dict(min_child=100, colsample=0.5, reg_lambda=5.0)),
    ("xendcg",       "otra pérdida de ranking (Adri)",                   dict(objective="rank_xendcg")),
])

**Conclusión bloque 2 · la CV no se mueve, pero `regularizado` y `lento` mejoran al peor controlador.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | Δ peor | |
|---|---|---|---|---|---|
| lento (`lr=0.03`, 700 árboles) | 0.900 | +0.001 | 0.763 | +0.018 | CV igual, más robusto |
| regularizado (`min_child=100`, `colsample=0.5`, `reg_lambda=5`) | 0.900 | 0.000 | **0.775** | **+0.030** | CV igual, más robusto |
| xendcg | 0.886 | **−0.013** | 0.740 | −0.006 | empeora |

- **En CV media ningún candidato gana:** con el futuro, el modelo base ya está bien ajustado.
- **El peor controlador sí mejora** con `regularizado` (+3 puntos) y `lento` (+1,8). Importa porque el leaderboard privado
  es **un solo controlador** (023) que no está en train: un modelo que no se hunda con ninguno es más seguro. Encaja con la
  hipótesis: obligarle a repartir entre columnas (`colsample=0.5`) evita que dependa solo de `f_next_input`.
- Ojo: con 1 semilla el peor controlador baila (en el bloque 1 iba de 0.740 a 0.757), así que hay que confirmarlo con 3 semillas.
- **`xendcg` se descarta.**

## 3 · Combinar lo que ha funcionado

Pon en `MEJOR` los cambios con veredicto **mejora** de los bloques 1 (p. ej. `feats=...` si sobra un bloque) y 2. Se comprueba con **3 semillas** frente a la base de 3 semillas
(sección 0), porque varios cambios que ayudan por separado no siempre suman.

In [ ]:
MEJOR = dict(
    # bloque 2: misma CV, mejor peor controlador -> se confirma con 3 semillas en la celda siguiente
    min_child=100, colsample=0.5, reg_lambda=5.0,     # regularizado
    lr=0.03, n_estimators=700,                        # lento
)

In [ ]:
# Comprobación con 3 semillas. Se puede saltar: el bloque 4 solo necesita la celda de MEJOR.
cv_comb = probar("comb/" + ",".join(f"{k}={v}" for k, v in MEJOR.items()), "combinación de ganadores", semillas=3, **MEJOR)
print(f"combinación: {cv_comb:.4f}   referencias: {REF}")

**Conclusión bloque 3 · la combinación no mejora: la mejora del peor controlador era ruido de 1 semilla.**

| (3 semillas) | CV | peor controlador | fold 0 | fold 1 | fold 2 | fold 3 | fold 4 |
|---|---|---|---|---|---|---|---|
| **`lgb_future`** (base) | **0.901** | **0.769** | 0.938 | 0.890 | 0.879 | 0.896 | 0.904 |
| `MEJOR` (regularizado + lento) | 0.901 | 0.766 | 0.938 | 0.894 | 0.884 | 0.885 | 0.904 |

- Misma CV y mismo peor controlador: con 3 semillas desaparece la ventaja que se veía con 1 (0.775 vs 0.746).
- Además tarda el doble (700 árboles). **Se queda `lgb_future` como versión final** y el bloque 4 no se lanza.
- Lectura general del barrido: con el futuro, los parámetros de referencia de `EQUIPO.md` ya están bien; el modelo depende de
  la información (bloque 1), no del ajuste.

## 4 · Versión para el stacking (`futuro/`)

Si la combinación mejora a la base, genera la versión definitiva con 3 semillas en el formato común. Nombre nuevo
(`lgb_future_v2`…) para no pisar `lgb_future`. Juanma la combina con `stack.py --futuro`.

In [ ]:
NOMBRE = "lgb_future_v2"
probar(NOMBRE, "versión stacking futuro: " + str(MEJOR), semillas=3, guardar=True, **MEJOR)

**Resultado:** no se lanza; `lgb_future` (CV 0.901, ya en `futuro/oof/` y `futuro/preds/`) es la versión para el stacking de esta rama.

## 5 · Análisis de errores y comparación con la red con futuro

**Pregunta:** ¿fallan el LightGBM y la red en los mismos snapshots? Si no, la combinación de Juanma ganará. Se mira el acierto
por tipo de tarea de cada modelo y en cuántos snapshots acierta uno solo.

In [ ]:
from metric import hits, labels

TIPOS = ["entry_conflict", "exit_conflict", "entry_coord", "exit_coord", "assume", "transfer", "conflict",
         "non_conformance", "qos", "entry_conditions", "exit_conditions", "aircraft_request", "zone", "return_to_route"]


def comparar(modelos=("lgb_future", "nn_future")):
    lab = labels().set_index("sample_id")
    tipos = pd.concat([pd.read_csv(ROOT / "Data" / f"{f}.csv", usecols=["sample_id", "target_index", "task_index"])
                       for f in ("train", "validation")]).query("target_index == 1").set_index("sample_id").task_index
    h = pd.DataFrame({m: hits(pd.read_csv(ROOT / "futuro" / "oof" / f"{m}.csv")) for m in modelos})
    a, b = modelos
    print(f"top-3:  {a} {h[a].mean():.3f}   {b} {h[b].mean():.3f}")
    print(f"aciertan los dos {(h[a] & h[b]).mean():.3f} | solo {a} {(h[a] & ~h[b].astype(bool)).mean():.3f} | "
          f"solo {b} {(~h[a].astype(bool) & h[b]).mean():.3f} | ninguno {((h[a] == 0) & (h[b] == 0)).mean():.3f}")
    h["tipo"] = tipos.reindex(h.index).map(dict(enumerate(TIPOS)))
    t = h.groupby("tipo").agg(muestras=(a, "size"), **{a: (a, "mean"), b: (b, "mean")})
    t["Δ (lgb − red)"] = t[a] - t[b]
    return t.sort_values("muestras", ascending=False).style.format(
        {a: "{:.2f}", b: "{:.2f}", "Δ (lgb − red)": "{:+.2f}"}).background_gradient(subset=["Δ (lgb − red)"], cmap="RdYlGn")


comparar(("lgb_future", "nn_future"))      # cambiar a lgb_future_v2 si se genera

**Conclusión del análisis · los dos modelos fallan casi en los mismos sitios; mezclarlos a partes iguales no suma.**

| | % de snapshots |
|---|---|
| aciertan los dos | 86,6 % |
| solo `lgb_future` | 3,5 % |
| solo `nn_future` | 2,2 % |
| fallan los dos | 7,7 % |

| Tipo correcto | muestras | `lgb_future` | `nn_future` | Δ |
|---|---|---|---|---|
| transfer | 3.527 | 0.98 | 0.98 | +0.00 |
| assume | 3.366 | 0.98 | 0.98 | +0.01 |
| zone_conflict | 2.121 | 0.92 | 0.90 | +0.02 |
| entry_coordination | 1.068 | 0.71 | 0.67 | +0.03 |
| return_to_route | 937 | 0.87 | 0.82 | +0.04 |
| exit_coordination | 745 | 0.57 | **0.59** | **−0.02** |
| exit_conditions | 448 | 0.80 | 0.76 | +0.04 |

- El LightGBM es igual o mejor en casi todos los tipos; la red solo gana en **exit_coordination**.
- Media de puntuaciones normalizadas (z-score dentro del snapshot), según el peso de la red: 0 → **0.901**, 0.3 → 0.900,
  0.5 → 0.897, 1 → 0.887. **Cuanto más peso a la red, peor:** con la información del futuro, los dos aprenden lo mismo.
- Para Juanma: con el futuro, `lgb_future` solo ya es lo mejor. Su meta-modelo (que puede fiarse de la red solo en
  exit_coordination) podría arañar algo, pero no hay que esperar mucho. Contraste con los modelos limpios, donde la red sí
  sumaba (LightGBM 0.648 → mezcla 0.655): sin futuro hay menos información y cada modelo captura cosas distintas.
- (Cifras calculadas con la misma función de la celda anterior; la salida de la celda no quedó guardada en el notebook.)

## 5b · Suavizado temporal de las predicciones

**Hipótesis:** la descripción de los datos dice que los snapshots consecutivos suelen compartir respuesta (el 72 % repite la del
anterior dentro de una racha contigua; rachas de 3 snapshots de mediana). Promediar la probabilidad de cada tarea con la de
la misma tarea en los snapshots vecinos (pesos que decaen con la distancia) debería corregir fallos aislados.

In [ ]:
import numpy as np, features_future as ff

_blk = ff.add_blocks(ff.add_folds(ff.load()))
_blk = _blk[_blk.split != "test"][["sample_id", "row_index", "task_name", "block", "blk_pos"]]


def suavizar(modelo, k=5, decay=0.7):
    """Media ponderada de la probabilidad de la misma tarea en los ±k snapshots contiguos."""
    x = _blk.merge(pd.read_csv(ROOT / "futuro" / "oof" / f"{modelo}.csv"), on=["sample_id", "row_index"])
    x["p"] = np.exp(x.score - x.groupby("sample_id").score.transform("max"))
    x["p"] /= x.groupby("sample_id").p.transform("sum")
    x = x.sort_values(["block", "task_name", "blk_pos"]); g = x.groupby(["block", "task_name"]).p
    num, den = x.p.copy(), pd.Series(1.0, index=x.index)
    for i in range(1, k + 1):
        for sh in (g.shift(i), g.shift(-i)):
            num += decay ** i * sh.fillna(0); den += decay ** i * sh.notna()
    return hits(x[["sample_id", "row_index"]].assign(score=x.p)).mean(), hits(x[["sample_id", "row_index"]].assign(score=num / den)).mean()


pd.DataFrame({m: suavizar(m) for m in ("lgb_future", "nn_future")}, index=["sin suavizar", "suavizado k=5"]).round(4)

**Conclusión 5b:** mejora real pero pequeña (calculado con esta función, k = 2 / 5 / 10):

| | sin suavizar | k=2 | k=5 | k=10 |
|---|---|---|---|---|
| `lgb_future` | 0.901 | 0.906 | 0.906 | **0.907** |
| `nn_future` | 0.887 | 0.895 | **0.898** | 0.898 |

- +0.6 puntos en el LightGBM, +1 en la red: las features de futuro ya contienen buena parte de esa información.
- Para usarlo en test haría falta aplicarlo también a `preds/` (mismas rachas en las sesiones de test). Candidato para la
  versión final de esta rama.

## 6 · Bloques nuevos: `coord` y `clear`

**Por qué:** en la sección 5 lo que queda son sobre todo las **coordinaciones** (entry + exit ≈ 45 % de los fallos) y
`f_next_input` no las ve: coordinar con un sector vecino no es una instrucción al avión. Los dos bloques nuevos miden,
para cada tarea, **cuántos segundos faltan hasta que cambia en la sesión el estado que esa tarea resuelve**:

| Bloque | Archivo | Columnas | Qué cambio futuro mide | Tipos a los que apunta |
|---|---|---|---|---|
| `coord` | `futuro/feat_coord.py` | `sc_*` | coordinación de entrada (`x_isc_*`), de salida (`k_out_*`), assume (`is_assumed` 0→1), transfer (1→0) | entry/exit_coordination, assume, transfer, gemelo assume/entry |
| `clear` | `futuro/feat_clear.py` | `sx_*` | nivel autorizado, rumbo, nivel de salida, fin de zona/conflicto, vuelta a ruta | exit_conditions, zone_conflict, return_to_route, conflictos |

**Hipótesis:** `coord` debería subir sobre todo entry/exit_coordination; `clear` los tipos de instrucciones. Se añaden de
uno en uno sobre la base (los 4 bloques de siempre), y después juntos. 1 semilla, ~1 min por variante.

In [ ]:
lanzar("nuevos", [
    ("+coord",       "cambios futuros de coordinación/assume/transfer",  dict(feats="base,logic,past,future,coord")),
    ("+clear",       "cambios futuros de instrucciones",                 dict(feats="base,logic,past,future,clear")),
    ("+coord+clear", "los dos bloques nuevos",                           dict(feats="base,logic,past,future,coord,clear")),
])

**Conclusión bloque 6 · `clear` mejora; `coord`, sorprendentemente, casi no.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | |
|---|---|---|---|---|
| +clear | **0.906** | **+0.006** | 0.757 | mejora (justo por encima del umbral) |
| +coord+clear | 0.905 | +0.006 | 0.775 | mejora |
| +coord | 0.901 | +0.001 | 0.769 | ruido |

- **`clear` es la que aporta:** sube 4 de los 5 folds (fold 0 +0.012, fold 2 +0.023). Saber cuándo cambia el nivel, el rumbo o
  la ruta del avión delata las tareas de instrucciones (exit_conditions, zone, return_to_route).
- **`coord` sola no mueve la CV**, aunque iba a por las coordinaciones. Lo más probable es que esa información ya estuviera en
  `f_task_end` (la tarea de coordinación desaparece de la lista justo cuando se coordina).
- `+coord+clear` empata con `+clear` en CV y tiene mejor peor controlador (0.775), pero con 1 semilla ese número baila
  (ya pasó en el bloque 2). La comparación por tipo de 6b dirá si `coord` aporta algo en las coordinaciones.
- **6b se lanza con `coord+clear`**: si por tipo `coord` no mejora las coordinaciones, la versión final se queda solo con `clear`.

### 6b · Versión con los bloques que ganen y dónde mejora

Poner en `FEATS_NUEVAS` la combinación ganadora del bloque 6. Se genera con 3 semillas como **`lgb_future_coord`**
(no pisa `lgb_future`) y se compara por tipo de tarea con `lgb_future`: así se ve si mejora justo donde se esperaba
(coordinaciones / instrucciones). ~4 min.

In [ ]:
FEATS_NUEVAS = "base,logic,past,future,coord,clear"     # bloque 6: +clear y +coord+clear mejoran (+0.006)
probar("lgb_future_coord", "versión stacking futuro: " + FEATS_NUEVAS, semillas=3, guardar=True, feats=FEATS_NUEVAS)
comparar(("lgb_future_coord", "lgb_future"))


=== lgb_future_coord versión stacking futuro: base,logic,past,future,coord,clear  seeds=3 feats=base,logic,past,future,coord,clear n_estimators=400 lr=0.05 leaves=31 min_child=50 subsample=0.8 colsample=0.7 trunc=5 reg_lambda=0.0 objective=lambdarank folds= quick=False
  fold 0 (37s)
  fold 1 (73s)
  fold 2 (109s)
  fold 3 (145s)
  fold 4 (181s)
lgb_future_coord: CV top3 = 0.9045   (min atco 0.772)   f0:0.950  f1:0.892  f2:0.887  f3:0.894  f4:0.903   147 features
Guardado: lgb_future_coord (oof) (preds + sub)
top-3:  lgb_future_coord 0.905   lgb_future 0.901
aciertan los dos 0.888 | solo lgb_future_coord 0.017 | solo lgb_future 0.014 | ninguno 0.082


,muestras,lgb_future_coord,lgb_future,Δ (lgb − red)
tipo,,,,
transfer,3527,0.98,0.98,+0.00
assume,3366,0.98,0.98,+0.00
zone,2121,0.92,0.92,-0.00
entry_coord,1068,0.73,0.71,+0.03
return_to_route,937,0.87,0.87,-0.00
exit_coord,745,0.59,0.57,+0.02
exit_conditions,448,0.82,0.80,+0.02
conflict,390,0.86,0.85,+0.01
qos,257,0.84,0.83,+0.01


**Conclusión 6b · ganancia pequeña pero real, y justo donde se esperaba.**

| 3 semillas | CV | peor controlador | fold 0 | fold 1 | fold 2 | fold 3 | fold 4 |
|---|---|---|---|---|---|---|---|
| `lgb_future` | 0.901 | 0.769 | 0.938 | 0.890 | 0.879 | 0.896 | 0.904 |
| **`lgb_future_coord`** | **0.905** | 0.772 | 0.950 | 0.892 | 0.887 | 0.894 | 0.903 |

| Tipo | `lgb_future` | `lgb_future_coord` | Δ |
|---|---|---|---|
| entry_coordination | 0.706 | 0.732 | **+0.026** |
| exit_coordination | 0.569 | 0.592 | **+0.023** |
| exit_conditions | 0.799 | 0.821 | **+0.022** |
| zone_conflict | 0.920 | 0.916 | −0.004 |
| return_to_route | 0.868 | 0.866 | −0.002 |
| non_conformance (86 muestras) | 0.895 | 0.791 | −0.105 |

- **+0.35 puntos** con 3 semillas (la mitad de lo que sugería 1 semilla); mejoran 3 de 5 folds.
- Las coordinaciones suben más de 2 puntos frente a `lgb_future`, **pero eso no demuestra que sea por `coord`**: la
  comparación es contra la base, no contra `+clear` solo. La sección 6c (con `+clear` solo a 3 semillas) indica que
  `coord` no aporta e incluso resta un poco.
- `clear` mejora exit_conditions, pero no zone_conflict ni return_to_route, a las que también apuntaba.
- non_conformance cae 10 puntos, pero son 86 muestras (~9 aciertos): probablemente ruido; vigilarlo.
- ~~`lgb_future_coord` pasa a ser el mejor LightGBM con futuro~~ → lo supera `lgb_future_clear` (0.907, sección 6c).
  `lgb_future_coord` es el mismo modelo que en el estudio de gpu.dsic se llamó `lgb_future_v2` (mismas cifras).
- Pendiente (norma de probar en las dos redes): añadir `sc_*` y `sx_*` a la red con futuro.

## 6c · Estudio en gpu.dsic (20/10, 21:35–22:20): otros árboles y confirmación de `clear`

Lanzado desde la terminal en gpu.dsic con `futuro/m_gbm_future.py` y `futuro/m_lgb_future.py`, mismos folds y features.
Referencia: `lgb_future` = CV 0.901 (3 semillas), 0.8996 (1 semilla). **En Kaggle (público, controlador 020): 0.942.**
Sus salidas y registros se generaron en gpu.dsic y no están en el repo: aquí solo quedan las cifras.

**1 · ¿Otro modelo de árboles mejora al LightGBM? → No** (mismas 125 features, 3 semillas)

| Modelo | CV top-3 | Peor controlador | Folds |
|---|---|---|---|
| **lgb_future** (referencia) | **0.901** | **0.769** | 0.938 / 0.890 / 0.879 / 0.896 / 0.904 |
| XGBoost `rank:ndcg` (600 árboles, depth 6, GPU) | 0.896 | 0.731 | 0.929 / 0.881 / 0.891 / 0.887 / 0.896 |
| CatBoost `YetiRank` (1500 iteraciones, depth 6, GPU) | 0.874 | 0.689 | 0.905 / 0.868 / 0.848 / 0.875 / 0.876 |

| Stacking (`stack.py --futuro`) | Mejor stack | Peor controlador |
|---|---|---|
| lgb + xgb | 0.9010 (los pesos dan 0 a xgb) | 0.769 |
| lgb + xgb + cat | 0.9012 (`stack_avg` por rangos) | 0.754 |

Correlación de aciertos: lgb–xgb 0.86, lgb–cat 0.73. Ni solos ni combinados mejoran al LightGBM, y empeoran al peor
controlador. Igual que con la red con futuro (sección 5): **con la información del futuro todos los modelos aprenden lo
mismo; el límite está en la información, no en el modelo.** XGBoost y CatBoost se descartan.

**2 · `clear` confirmado con 3 semillas**

| Modelo | CV | Δ vs 0.901 | Peor controlador | Folds |
|---|---|---|---|---|
| **`lgb_future_clear`** (+clear) | **0.9072** | **+0.006** | 0.769 | 0.949 / 0.893 / 0.894 / 0.897 / 0.905 |
| `lgb_future_coord` (+coord +clear, sección 6b) | 0.9045 | +0.004 | 0.772 | 0.950 / 0.892 / 0.887 / 0.894 / 0.903 |

- **`clear` funciona:** +0.6 puntos, mejora o iguala en los 5 folds y no empeora al peor controlador.
- **`coord` no aporta, e incluso resta un poco** combinado con `clear`. Las coordinaciones (la mitad de los fallos que
  quedan) siguen sin verse. Hipótesis: el cambio de estado de coordinación ocurre lejos en el tiempo de cuando el
  controlador trabaja en ella, o cae en huecos sin snapshots etiquetados.
- **Modelo elegido: `lgb_future_clear` (CV 0.907)**, base del segundo nivel temporal (suavizado, sección 7)
  y del envío de las ~00:00.

**3 · Notas técnicas de gpu.dsic:** 2× Quadro RTX 5000 (16 GB), 32 núcleos. LightGBM va en CPU (8 hilos): caben 4 pruebas
a la vez. XGBoost 3.x de pip no trae GPU allí → `xgboost==2.0.3` con `task_index` en one-hot (su camino categórico falla
con numpy 2). CatBoost YetiRank en GPU calcula la métrica PFound en CPU: se calcula solo al final (`metric_period`).

## 7 · Suavizado temporal con test y CV anidada (Juanma)

Completa la 5b (mismo método `prob`, ahora con CV anidada y aplicado a test). Hay un snapshot cada ~3 s y el controlador suele seguir con la misma tarea varios seguidos, así que se mezcla
el score de cada tarea con el de la **misma tarea** (`session`, `task_name`) en los ±k snapshots de la misma racha (`block`):

- **prob** (el de la 5b, de Pablo): `(p(t) + Σ_j decay^j · p(t±j)) / (1 + Σ_j decay^j · [la tarea está en ese vecino])`, con `p` = softmax del score dentro del snapshot.
- **log** (alternativa): `lsm(t) + a · Σ_j decay^(j-1) · [lsm(t-j) + lsm(t+j)]`, con `lsm` = log-softmax.

Se comparan los dos con CV anidada y se usa el que gane para cada modelo.

Diferencias con la 5b:
- **CV anidada**: `a`, `decay` y `k` se eligen con 4 folds y se miden en el 5º. Así el número no está inflado por elegir entre muchas combinaciones.
- **Se aplica también a test** (`preds/`), con los vecinos de las sesiones de test. No usa etiquetas, así que no hay fuga. Los vecinos son de la misma sesión, por tanto del mismo controlador y fold, y su OOF tampoco vio la etiqueta.
- Las salidas van a `futuro/pruebas/oof|preds/suav_<modelo>.csv` y salen en `futuro/pruebas/panel_pruebas.ipynb`.

Código en `futuro/pruebas/suavizado.py`.

In [ ]:
import itertools
sys.path.insert(0, str(ROOT / "futuro" / "pruebas"))
import suavizado as S
from base_fut import datos, guardar, registrar

MODELOS = ["lgb_future", "nn_future"]          # cualquiera con oof (y preds) en futuro/ o futuro/pruebas/
GRIDS = {"prob": list(itertools.product((0.5, 0.7, 0.85, 1.0), (1, 2, 3, 5, 10))),                       # (decay, k)
         "log": list(itertools.product((0.1, 0.2, 0.3, 0.5, 0.75, 1.0), (1.0, 0.7, 0.5), (1, 2, 3, 5, 10)))}  # (a, decay, k)

_lab, _te, _ = datos()
_cols = ["sample_id", "row_index", "session", "block", "t", "task_name"]
LS = _lab[_cols + ["target_index", "fold"]].assign(y=_lab.target_index)
TS = _te[_cols].copy()
NB_L, NB_T = S.vecinos(LS, 10), S.vecinos(TS, 10)   # posición de la misma tarea en los ±10 snapshots vecinos

In [ ]:
def aplicar(metodo, score_lsm, nb, g):
    return S.suavizar_prob(np.exp(score_lsm), nb, *g) if metodo == "prob" else S.suavizar(score_lsm, nb, *g)


def suavizado_anidado(modelo):
    L = LS.merge(S.leer(modelo, "oof"), on=["sample_id", "row_index"], how="left")
    base = S.lsm(L, L.score.values)
    n, mejor = L.sample_id.nunique(), None
    for metodo, grid in GRIDS.items():
        sc = {g: aplicar(metodo, base, NB_L, g) for g in grid}
        filas, tot0, tot1 = [], 0, 0
        for f in range(5):
            tr, va = (L.fold != f).values, (L.fold == f).values
            g = max(grid, key=lambda g: S.hit3(L, sc[g], tr))
            h0, h1, m = S.hit3(L, base, va), S.hit3(L, sc[g], va), L[va].sample_id.nunique()
            tot0, tot1 = tot0 + h0 * m, tot1 + h1 * m
            filas.append(dict(fold=f, params=g, sin=h0, con=h1, mejora=h1 - h0))
        g_final = max(grid, key=lambda g: S.hit3(L, sc[g]))
        print(f"{modelo} · {metodo}: CV anidada {tot0 / n:.4f} -> {tot1 / n:.4f} ({(tot1 - tot0) / n:+.4f}); "
              f"parámetros finales (todo el OOF): {g_final}")
        display(pd.DataFrame(filas).round(4))
        if mejor is None or tot1 > mejor[0]:
            mejor = (tot1, metodo, g_final)
    print(f"-> {modelo}: se usa {mejor[1]} con {mejor[2]}\n")
    return L, base, mejor[1], mejor[2]


SUAV = {m: suavizado_anidado(m) for m in MODELOS}

In [ ]:
# aplica el método y los parámetros finales a OOF y test, guarda suav_<modelo> y lo registra para el panel
for modelo, (L, base, metodo, g) in SUAV.items():
    p = S.leer(modelo, "preds")
    T = TS.merge(p, on=["sample_id", "row_index"], how="left") if p is not None else None
    so = aplicar(metodo, base, NB_L, g)
    sp = aplicar(metodo, S.lsm(T, T.score.values), NB_T, g) if T is not None else None
    guardar(f"suav_{modelo}", L, so, T, sp)
    registrar(f"suav_{modelo}", hilo="suavizado", n_features=None,
              descripcion=f"{modelo} suavizado {metodo} {g} (notebook LGB_futuro)",
              extra={"metodo": metodo, "params": list(g)})
    print(f"suav_{modelo}: guardado", "con preds de test" if sp is not None else "SIN preds (no hay preds del modelo)")

**Conclusión 7** (CV anidada, primera medida):

| | sin suavizar | log | **prob (5b)** |
|---|---|---|---|
| `lgb_future` | 0.9010 | 0.9033 (+0.0023) | **0.9042 (+0.0032)** |
| `nn_future` | 0.8874 | 0.8776 (−0.0097) | **0.8973 (+0.0099)** |

- El método de la 5b (media de probabilidades normalizada) es el bueno. En la red el log empeora, porque sus scores tienen otra escala.
- Los 0.906–0.907 de la 5b salían de elegir k mirando todo el OOF. En CV anidada la mejora real es +0.3 puntos en el LightGBM y +1 en la red.
- Con preds de test ya se puede usar: siguiente paso, suavizar el stack de futuro.

## 9 · Segundo paso de sesión: «cada tarea se trabaja una sola vez» (20/10, 22:40–23:15 · gpu.dsic)

**Análisis de errores de `lgb_future_clear` (0.9072)** que lleva a la idea:

| Tipo correcto | muestras | acierto | % de los fallos |
|---|---|---|---|
| exit_coordination | 745 | 0.596 | 24.6 % |
| entry_coordination | 1068 | 0.735 | 23.1 % |
| zone_conflict | 2121 | 0.922 | 13.5 % |
| return_to_route | 937 | 0.877 | 9.4 % |
| exit_conditions | 448 | 0.812 | 6.9 % |

- **Las coordinaciones casi no dejan huella:** en exit_coordination, cuando no hay siguiente instrucción al avión
  (`f_next_act` vacío, 241 muestras) el acierto es 0.21, y `sc_task` está vacío en la mitad. Un barrido de *todas* las
  columnas del avión (¿cambia en ≤ 60 s?) solo encuentra señales que ya están en `f_*`/`sx_*` (nivel autorizado, etc.).
- **Los fallos son de tramo, no de snapshot:** el 74 % de los fallos están en tramos (racha, tarea) en los que el modelo
  no acierta en ningún snapshot. Por eso el suavizado dentro de la racha apenas suma.
- **Estructura de las etiquetas** (lo que no estábamos usando):
  - **el 98.2 % de las tareas son la respuesta en un solo episodio de la sesión**;
  - tras un hueco entre rachas la respuesta cambia casi siempre (solo el 0.2 % repite);
  - el 87 % de las rachas tienen una sola respuesta;
  - test tiene la misma estructura: 401 rachas, 4.6 snapshots por racha (train: 4.2).

**Hipótesis:** si el primer pase está muy seguro de que la tarea X fue la respuesta en *otra* racha de la sesión, aquí
probablemente no lo es (*explaining away*). No usa etiquetas: solo los scores OOF del primer pase en otras rachas de la
misma sesión, que están en el mismo fold.

**Variantes, todas sobre el OOF de `lgb_future_clear` (0.9072):**

| Variante | CV | peor controlador | |
|---|---|---|---|
| regla `lsm − λ·M` (M = máx. prob. media de la tarea en otra racha), λ elegido en CV anidada | 0.9135 | — | los 5 folds eligen λ=2 y mejoran |
| + pooling en la racha + 2–3 iteraciones (CV anidada) | 0.9135 | — | no añade |
| 2.º pase LightGBM (todas las features + 16 de sesión) | 0.9055 | 0.719 | **empeora**: se apoya en el score del 1.er pase y sobreajusta |
| **logit condicional con 9 features de sesión** (`futuro/feat_session.py`) | **0.9142** | **0.781** | pesos casi iguales en los 5 folds (M ≈ −2.6) |
| logit, 2 / 3 rondas | 0.9128 / 0.9114 | 0.781 | iterar empeora |

El mismo logit sobre otros primeros pases: `lgb_future` 0.901 → 0.911, `lgb_future_coord` 0.9045 → 0.912; la red
`nn_future` 0.887 → 0.878 (empeora, como con el suavizado `log`: sus scores tienen otra escala).

**Por tipo** (logit sobre `lgb_future_clear`): sube todo menos transfer (igual). entry_coordination +1.9,
exit_coordination +1.5, return_to_route +1.2, exit_conditions +1.8, qos +3.9, exit_conflict +5.1. Por controlador:
mejoran 15 de 18; la peor bajada es de −0.005 (atco 4).

Se usa desde la terminal con `--sesion`: hace el primer pase y el segundo, y guarda `<name>` y `<name>_ses`.
En CV, los pesos del logit se ajustan con los otros 4 folds; para test, con todo el OOF, y se aplican a `preds/`.

In [ ]:
# Estructura de las etiquetas que justifica el segundo paso (cifras de la tabla de arriba). Solo usa train+validation.
lab = data["lab"]
y = lab[lab.target_index == 1].sort_values(["session", "t"])
y["seg"] = ((y.task_name != y.groupby("session").task_name.shift()) | (y.blk_pos == 0)).cumsum()   # tramo = racha + tarea
seg = y.groupby("seg").agg(session=("session", "first"), task=("task_name", "first"), block=("block", "first"))
n_seg = seg.groupby(["session", "task"]).size()
print(f"tareas que son respuesta en un solo tramo de la sesión: {(n_seg == 1).mean():.3f}")
print(f"rachas con una sola respuesta: {(seg.groupby('block').size() == 1).mean():.3f}")
s0 = y[y.blk_pos == 0]
print(f"primer snapshot de una racha con la misma respuesta que la racha anterior: "
      f"{(s0.task_name == y.groupby('session').task_name.shift().loc[s0.index]).mean():.3f}")
te = data["te"].drop_duplicates("sample_id")
print(f"test: {len(te)} snapshots en {te.block.nunique()} rachas ({len(te) / te.block.nunique():.1f} por racha)")

In [ ]:
# ~3 min por fold con 3 semillas; guarda lgb_future_clear y lgb_future_clear_ses (oof, preds, submission)
probar("lgb_future_clear", "3s +clear", semillas=3, guardar=True, feats="base,logic,past,future,clear", sesion=True)

**Conclusión 9 · el segundo paso de sesión es la mayor mejora desde `clear`: +0.7 puntos, y sube al peor controlador.**

Ejecución limpia desde la terminal (gpu.dsic, 3 semillas, `--sesion`; registro en `experiments/lgb_future_runs.csv`):

| 3 semillas | CV | peor controlador | fold 0 | fold 1 | fold 2 | fold 3 | fold 4 |
|---|---|---|---|---|---|---|---|
| `lgb_future_clear` (1.er pase, regenerado) | 0.9071 | 0.769 | 0.949 | 0.893 | 0.894 | 0.897 | 0.905 |
| **`lgb_future_clear_ses`** | **0.9142** | **0.781** | 0.952 | 0.897 | 0.905 | 0.906 | 0.913 |

- Mejoran los 5 folds (+0.3 a +1.1) y el peor controlador (+1.2).
- Pesos ajustados con todo el OOF (los que se aplican a test): `lsm` 0.79, `lPb` 0.36, **`M` −2.67**, `E` 0.99,
  **`Pnext` −1.44**, `Pc` 0.26, `Mc` −0.23, `top_else` −0.08, `Pprev` 0.00. Lo que pesa es "la tarea ya fue la
  favorita en otra racha" (M) y "es la favorita en la racha siguiente" (Pnext): como tras un hueco la respuesta
  cambia, si en la racha siguiente gana esta tarea, aquí probablemente no es.
- Es complementario al suavizado (sección 7), que solo mira dentro de la racha; el pooling en la racha ya lo cuenta `lPb`.
- Salidas: `futuro/oof|preds/lgb_future_clear_ses.csv` y `futuro/submissions/sub_lgb_future_clear_ses.csv`
  (comprobada: 1863 snapshots, 3 tareas cada uno). `lgb_future_clear.csv` se ha regenerado (0.9071 frente a 0.9072:
  ruido de los hilos).
- ~~**Para el envío:** `lgb_future_clear_ses`~~ → superado por `lgb_future_clear10_ses` + suavizado (9b).

### 9b · Confirmaciones y variantes del segundo paso (gpu.dsic, 3 pruebas en paralelo × 8 hilos)

Lanzado desde la terminal con `--sesion`, p. ej.
`python futuro/m_lgb_future.py --feats base,logic,past,future,clear --sesion --seeds 10 --name lgb_future_clear10` (registro en `experiments/lgb_future_runs.csv`, logs en `logs/`). El 1.er pase y el
2.º se miden igual: misma CV con `Data/folds.csv`.

| Prueba | semillas | 1.er pase | + sesión | peor controlador (+ sesión) | |
|---|---|---|---|---|---|
| `clear` (referencia, `sesion/base`) | 1 | 0.9057 | 0.9132 | 0.778 | |
| `clear`, `colsample=0.5` (`sesion/colsample05`) | 1 | 0.9066 | 0.9125 | 0.775 | ruido: un 1.er pase "más repartido" no deja más margen al 2.º |
| `lgb_future_coord` (+coord +clear) | 3 | 0.9045 | 0.9123 | 0.772 | `coord` tampoco ayuda con sesión |
| `lgb_future_clear` | 3 | 0.9071 | 0.9142 | 0.781 | sección 9 |
| **`lgb_future_clear10`** | **10** | **0.9086** | **0.9158** | **0.781** | más semillas: +0.15 en los dos pasos |

**Features de sesión extra para las coordinaciones (prototipo sobre el OOF de `lgb_future_clear`, logit por fold):**

| Variante | CV | peor controlador | entry_coord | exit_coord |
|---|---|---|---|---|
| segundo paso de la sección 9 | 0.9142 | 0.781 | 0.754 | 0.611 |
| + gemelo assume↔entry_coordination y exit_coordination↔transfer (máx. prob. del gemelo en una racha anterior/posterior) | 0.9135 | 0.781 | 0.740 | 0.604 |
| + pesos propios para coordinaciones (`M`, `lsm`, `lPb` × [tipo 2 o 3]) | 0.9141 | 0.793 | 0.762 | 0.613 |
| + las dos | 0.9145 | 0.784 | 0.750 | 0.628 |

Todo dentro del ruido: no se añade (prototipo hecho fuera del repo; no se conserva su código porque se descarta). Lo que ya cuenta `Mc` (avión) recoge lo del gemelo.

**Suavizado temporal (sección 7, método `prob`) ENCIMA del segundo paso, en CV anidada:**

| Primer + segundo paso | sin suavizar | suavizado (CV anidada) | parámetros por fold | peor controlador |
|---|---|---|---|---|
| `lgb_future_clear_ses` (3 semillas) | 0.9142 | 0.9165 (+0.23) | (0.7,3), (0.85,5) ×4 | — |
| **`lgb_future_clear10_ses`** (10 semillas) | 0.9158 | **0.9181 (+0.23)** | (0.85,3) ×4, (1.0,3) | **0.784** |

- Ahora el suavizado suma algo más que sobre `lgb_future_clear` (+0.1). Tiene sentido: el 2.º paso mira *otras* rachas,
  y el suavizado, los snapshots vecinos dentro de la misma.
- Parámetros finales (elegidos con todo el OOF): decay=0.85, k=3, los mismos que en la sección 7. Con ellos la CV es 0.9183
  (optimista); el número honesto es el de la CV anidada, 0.918.
- Salidas: `futuro/oof|preds/lgb_future_clear10[_ses].csv`, `futuro/submissions/sub_lgb_future_clear10_ses.csv`;
  suavizado en `futuro/pruebas/oof|preds/suav_lgb_future_clear10_ses.csv` y
  `futuro/submissions/sub_suav_lgb_future_clear10_ses.csv` (comprobada: 1863 snapshots, 3 tareas cada uno).

**Candidatas para el envío (las elige Juanma):**
1. `sub_suav_lgb_future_clear10_ses.csv`: CV anidada **0.918**, peor controlador 0.784 (la mejor).
2. `sub_lgb_future_clear10_ses.csv`: CV 0.916, peor controlador 0.781 (sin suavizado: un paso menos).

In [ ]:
# Suavizado prob encima del segundo paso, con CV anidada; aplica los parámetros finales a test y guarda
# futuro/pruebas/oof|preds/suav_<modelo>.csv y futuro/submissions/sub_suav_<modelo>.csv (~2 min).
import itertools, numpy as np
sys.path.insert(0, str(ROOT / "futuro" / "pruebas"))
import suavizado as S
from base_fut import guardar

MODELO = "lgb_future_clear10_ses"
cols = ["sample_id", "row_index", "session", "block", "t", "task_name"]
L = data["lab"][cols + ["target_index", "fold"]].merge(pd.read_csv(ROOT / "futuro" / "oof" / f"{MODELO}.csv"), how="left")
L["y"] = L.target_index
T = data["te"][cols].merge(pd.read_csv(ROOT / "futuro" / "preds" / f"{MODELO}.csv"), how="left")
NB, NBT = S.vecinos(L, 10), S.vecinos(T, 10)
base = S.lsm(L, L.score.values)
grid = list(itertools.product((0.5, 0.7, 0.85, 1.0), (1, 2, 3, 5, 10)))       # (decay, k)
sc = {g: S.suavizar_prob(np.exp(base), NB, *g) for g in grid}
filas = []
for f in range(5):
    tr, va = (L.fold != f).values, (L.fold == f).values
    g = max(grid, key=lambda g: S.hit3(L, sc[g], tr))
    filas.append(dict(fold=f, params=g, n=L[va].sample_id.nunique(), sin=S.hit3(L, base, va), con=S.hit3(L, sc[g], va)))
t = pd.DataFrame(filas)
print(f"CV anidada {(t.sin * t.n).sum() / t.n.sum():.4f} -> {(t.con * t.n).sum() / t.n.sum():.4f}"); display(t.round(4))
g = max(grid, key=lambda g: S.hit3(L, sc[g]))
sp = S.suavizar_prob(np.exp(S.lsm(T, T.score.values)), NBT, *g)
guardar("suav_" + MODELO, L, sc[g], T, sp)
x = T[["sample_id", "row_index"]].assign(s=sp).sort_values(["sample_id", "s"], ascending=[True, False])
x.groupby("sample_id").row_index.apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction").reset_index() \
    .to_csv(ROOT / "futuro" / "submissions" / f"sub_suav_{MODELO}.csv", index=False)
print("parámetros finales:", g)

## 10 · Siguientes ideas (20/10, 23:20)

Objetivo: pasar de 0.947 a ≥ 0.957 en Kaggle público. Se decide por la CV (y el peor controlador), no por el público.
Ya hecho: segundo paso de sesión (9), 10 semillas y suavizado encima (9b). Probado sin éxito (9b): features del gemelo,
pesos propios para las coordinaciones, `colsample=0.5`, `coord` con sesión. Pendientes que ya estaban: `clear` en la red
con futuro; las coordinaciones siguen sin señal propia.

Las ideas nuevas y sus resultados van en las secciones 11 y siguientes.


## 11 · Más allá del segundo paso (20/10, 23:30–00:00 · gpu.dsic)

**Referencia común:** `clear` con 1 semilla = 0.9057 en el 1.er pase y **0.9132 con el segundo paso** (`sesion/base`).
Sobre 10 semillas: 0.9086 → 0.9158. Pruebas rápidas desde la terminal (`--quick --sesion`) o prototipos sobre el OOF.

### 11a · Features de futuro por racha (`futuro/feat_block.py`, bloque `block`, columnas `bk_*`)

**Hipótesis:** los fallos son de tramo (sección 9), pero `f_*`/`sx_*`/`sc_*` se miden desde cada snapshot. Para cada
(racha, tarea) se toma el evento más temprano visto desde cualquier snapshot de la racha, medido respecto al **final de
la racha**: siguiente instrucción, 2.ª instrucción, desaparición de la tarea/avión, `sx_task`, `sc_task`; más el
tipo de esa instrucción y cuántas instrucciones recibe el avión durante la racha.

La señal existe: la tarea correcta desaparece justo al acabar la racha (mediana de `bk_f_task_end` 0 s, frente a 297 s
en el resto), y su siguiente instrucción, vista desde la racha, llega a 2 s del final (11 s vista desde el snapshot).

| 1 semilla | 1.er pase | + sesión | peor controlador (+ sesión) |
|---|---|---|---|
| referencia `clear` | 0.9057 | **0.9132** | 0.778 |
| `+block` | 0.9083 | 0.9116 | 0.772 |
| `+block+coord` | 0.9094 | 0.9129 | 0.769 |

**Conclusión:** el 1.er pase sube +0.3, pero tras el segundo paso no queda nada: lo que aportan ya lo recoge el
pooling por racha del segundo paso (`lPb`). **No se usa** (el bloque se queda en el código por si sirve a la red).

### 11b · Otras formas del segundo paso (prototipos sobre el OOF de `lgb_future_clear10`, 0.9086)

| Variante | CV | peor controlador | |
|---|---|---|---|
| **segundo paso actual** (logit de 9 pesos) | **0.9158** | 0.781 | |
| asignación global tipo Sinkhorn (cada racha suma 1, cada tarea ≤ 1 en la sesión), CV anidada | 0.9102 | 0.775 | peor: las restricciones duras no encajan |
| + `M` separada en rachas cercanas (≤ 60 s) y lejanas | 0.9159 | 0.781 | ruido |
| + probabilidad en la racha +2 | 0.9159 | 0.781 | ruido |
| + `Pnext` solo si el hueco hasta la siguiente racha es < 30 s | 0.9159 | 0.781 | ruido |
| + `lPb` × log(longitud de la racha) | 0.9158 | 0.784 | ruido |
| + todas | 0.9158 | 0.784 | ruido |

**Conclusión:** el segundo paso está saturado con lo que da la estructura de sesión.

### 11c · Pseudo-etiquetas del controlador nuevo

**Hipótesis:** el privado es un solo controlador que no está en train. Reentrenar el 1.er pase añadiendo los snapshots
del propio fold de validación, etiquetados con el top-1 de `lgb_future_clear10_ses` cuando su probabilidad > τ, adapta
el modelo a su estilo. Sin fuga: las pseudo-etiquetas del fold f salen de modelos que no vieron f.

| 1 semilla | pseudo-etiquetados | acierto de la pseudo-etiqueta | 1.er pase | + sesión | peor controlador |
|---|---|---|---|---|---|
| referencia | — | — | 0.9057 | 0.9132 | 0.778 |
| τ = 0.8 | 66 % | 0.963 | 0.9045 | 0.9138 | 0.787 |
| τ = 0.5 | 84 % | 0.898 | 0.9053 | 0.9129 | 0.787 |

**Conclusión:** ruido en la CV. El peor controlador sube un punto, pero con 1 semilla eso baila. No se usa.

### 11d · ¿Qué queda? Análisis de errores de `lgb_future_clear10_ses` (0.9158)

| Tipo correcto | muestras | acierto | % de los fallos |
|---|---|---|---|
| exit_coordination | 745 | 0.624 | 25 % |
| entry_coordination | 1068 | 0.746 | 24 % |
| zone_conflict | 2121 | 0.933 | 13 % |
| return_to_route | 937 | 0.892 | 9 % |
| exit_conditions | 448 | 0.835 | 7 % |

- En las entry_coordination falladas, la favorita suele ser **otra entry_coordination** (92 de 271): el modelo no sabe
  cuál de los aviones se coordina. En las exit_coordination falladas, el 64 % no tienen ninguna instrucción futura.
- **Solo el 10–13 % de las tareas de coordinación se trabajan alguna vez en la sesión** (assume: 90 %, transfer:
  96 %, zone_conflict: 68 %). No hay una restricción de "todas se hacen" que explotar.
- Que la tarea desaparezca antes que su avión ("se resuelve") **no predice** qué coordinación se trabaja
  (exit_coordination: ~10 % trabajadas en todos los tramos).
- **Conclusión:** las coordinaciones casi no dejan rastro en los datos; es el techo de la información disponible.

### 11e · Peso por tramo en el 1.er pase y combinación de primeros pases (20/10, 23:40)

**Peso por tramo** (`--peso racha|tramo --peso-pot p`): cada snapshot pesa 1/longitud^p, para que las rachas largas (muchos
snapshots casi iguales) no dominen el entrenamiento. `racha` = snapshots de la racha (sin etiqueta); `tramo` = snapshots
seguidos con la misma respuesta (usa la etiqueta, solo de las filas de entrenamiento). 1 semilla, 4 pruebas a la vez.

| 1 semilla | 1.er pase | + sesión | peor controlador (+ sesión) |
|---|---|---|---|
| referencia (sin peso) | 0.9057 | **0.9132** | 0.778 |
| `racha`, p = 0.5 | 0.9048 | 0.9097 | 0.778 |
| `racha`, p = 1 | 0.9069 | 0.9124 | 0.775 |
| `tramo`, p = 0.5 | 0.9064 | 0.9110 | 0.790 |
| `tramo`, p = 1 | 0.9055 | 0.9132 | 0.757 |

**Conclusión:** ruido o peor. No se usa (la opción queda en el código, apagada por defecto).

**Combinación de primeros pases antes del segundo paso** (z-score de cada modelo dentro del snapshot; referencia
`lgb_future_clear10` en z-score: 0.9086 → 0.9151 con sesión):

| + otro modelo | peso | 1.er pase | + sesión | peor controlador |
|---|---|---|---|---|
| `xgb_future` | 0.20 / 0.35 / 0.50 | 0.9085 / 0.9064 / 0.9035 | 0.9139 / 0.9117 / 0.9113 | 0.778 / 0.766 / 0.766 |
| `lgb_future_coord` | 0.20 / 0.35 | 0.9085 / 0.9073 | 0.9157 / 0.9143 | 0.790 / 0.790 |

**Conclusión:** lo mismo que en 6c (stacking): mezclar con otros modelos no suma, tampoco con el segundo paso encima. La
red y CatBoost no se lanzaron (ya eran peores en 5 y 6c).

## 12 · Parámetros con el segundo paso y el oráculo (20/10 23:45 – 21/10 00:30 · gpu.dsic)

### 12a · Parámetros del 1.er pase con el segundo paso encima

**Hipótesis:** los parámetros se eligieron sin el segundo paso. Ahora importa también el orden de las posiciones 2–10
(de ahí salen `M`, `Pnext`), y `trunc=5` solo optimiza las 5 primeras. 1 semilla, `--quick --sesion`, 3 a la vez.

| 1 semilla, + sesión | CV | peor controlador | |
|---|---|---|---|
| referencia (`leaves=31`, `min_child=50`, `trunc=5`, 400 árboles) | 0.9132 | 0.778 | |
| `leaves=15` | 0.9123 | 0.754 | ruido |
| `leaves=63` | 0.9151 | 0.760 | ruido |
| `min_child=20` | 0.9136 | 0.787 | ruido |
| `min_child=100` | 0.9104 | 0.754 | peor |
| `trunc=10` / `trunc=20` | 0.9135 / 0.9121 | 0.769 / 0.763 | ruido |
| 800 árboles, `lr=0.025` | 0.9138 | 0.787 | ruido |
| **`leaves=63`, `min_child=100`** | **0.9174** | 0.775 | mejora con 1 semilla → se confirma |

Confirmación con 3 semillas (`lgb_future_clear_l63`): 1.er pase 0.9076 (frente a 0.9071), **+ sesión 0.9159** (frente a
0.9142), peor controlador 0.766 (frente a 0.781). **No se confirma** (+0.17, ruido, y peor controlador más bajo): se
quedan los parámetros de siempre.

### 12b · El oráculo: qué hicieron otros controladores en el mismo escenario (`futuro/feat_oracle.py`, `--oraculo`)

**De dónde sale:** idea Pau #1 del EDA (`notebooks/01_eda.ipynb`, sección 9). En `main` se descartó **por la regla 4**
(usa `scenario` y `timestamp` para alinear), **no porque se midiera**: aquí se mide por primera vez.

**Por qué puede funcionar:** las sesiones de un escenario tienen el mismo tráfico. Los aviones de las 5 sesiones de
test están al 100 % en sesiones etiquetadas del mismo escenario (11–14 por escenario). El EDA ya veía que la mezcla de
tipos elegidos depende más del escenario (Cramér's V 0.159) que del controlador (0.096). Y ataca nuestro hueco: solo el
10–13 % de las coordinaciones se trabajan (11d), y si hay que hacer una de verdad, otros controladores también la hacen.

**Cómo se calcula:**
- **Tiempo alineado:** segundos desde el inicio de la sesión, corregidos con el desfase mediano de la primera
  aparición de los aviones comunes del escenario (sin etiquetas). Entre sesiones, un mismo avión aparece con ~22 s de
  diferencia.
- **Features `or_*`:** fracción de las otras sesiones en que esta tarea fue respuesta alguna vez (`or_ever`), a ±60 s
  y ±180 s del mismo momento (`or_near60/180`), lo mismo para cualquier tarea del avión (`or_cs_near60`), segundos
  hasta el momento mediano en que la trabajan los demás (`or_dt`, `or_dt_abs`) y rangos en el snapshot.
- **Sin fuga (usa etiquetas de OTRAS sesiones):** para el modelo del fold f solo cuentan sesiones etiquetadas del
  mismo escenario, de folds ≠ f y **de otro controlador que la fila** (también para las filas de entrenamiento).
  Para el modelo final y test: todas las sesiones etiquetadas salvo las del propio controlador.
- La integración en `m_lgb_future.py` reproduce exactamente el prototipo (`futuro/pruebas/oraculo.py`): 0.9070 /
  0.9192 con la misma semilla.

**Resultados:**

| | semillas | 1.er pase | + sesión | peor controlador (+ sesión) | folds (+ sesión) |
|---|---|---|---|---|---|
| sin oráculo | 1 | 0.9057 | 0.9132 | 0.778 | 0.955 / 0.893 / 0.903 / 0.903 / 0.916 |
| **con oráculo** | 1 | 0.9070 | **0.9192** | 0.784 | 0.954 / 0.905 / 0.899 / 0.916 / 0.925 |
| sin oráculo (`lgb_future_clear_ses`) | 3 | 0.9071 | 0.9142 | 0.781 | 0.952 / 0.897 / 0.905 / 0.906 / 0.913 |
| **con oráculo** | 3 | 0.9091 | **0.9236** | **0.763** ⚠️ | 0.954 / 0.905 / 0.908 / 0.923 / 0.931 |
| con oráculo + `leaves=63`, `min_child=100` | 3 | 0.9088 | 0.9211 | 0.760 | 0.958 / 0.902 / 0.903 / 0.925 / 0.923 |

- **+0.9 puntos con 3 semillas, y suben los 5 folds.** Casi todo aparece **en el segundo paso**: el oráculo apenas
  mueve el 1.er pase (+0.2), pero le da al segundo paso mejores candidatos en las posiciones 2–10.
- **Por tipo (1 semilla, + sesión):** exit_coordination 0.608 → **0.663 (+5.5)**, entry_coordination 0.741 → **0.767
  (+2.6)**, qos +3.1, exit_conflict +2.0, non_conformance +3.5. Bajan zone_conflict −0.8 y conflict_resolution −1.0.
  Mejora justo donde se esperaba: la mitad de los fallos eran coordinaciones.
- **Ganancia en el árbol:** `or_near180` 9 % del total, `or_dt` 3 %, `or_ever` 2 %; las demás, < 1 %.
- ⚠️ **El peor controlador baja con 3 semillas** (0.781 → 0.763). El privado es un solo controlador: se analiza abajo.

**Variantes del oráculo** (1 semilla, + sesión; base = 0.9192 / 0.784):

| Variante | Qué añade | CV | peor controlador |
|---|---|---|---|
| B | ventanas ±30/300/600 s, `or_cs_ever`, núcleo exp(−|Δt|/120) | 0.9187 | 0.763 |
| C | ventanas en tiempo relativo a la primera aparición del avión | 0.9175 | 0.737 |
| D | solo `or_near180`, `or_dt`, `or_ever` | 0.9188 | 0.749 |
| E | B + C | 0.9194 | 0.784 |

**Conclusión:** ninguna variante mejora a la base. **Se queda la base.**

**⚠️ Antes de subirlo (decisión del equipo):** el oráculo usa `scenario` y `timestamp` para alinear con otros
controladores (regla 4) y las etiquetas de otras sesiones. La CV es honesta (folds por controlador; el 023 vuela los
mismos escenarios), pero hay que poder defenderlo ante el jurado.

In [ ]:
# ~6 min con 3 semillas: guarda lgb_future_clear_or y lgb_future_clear_or_ses (oof, preds, submission)
probar("lgb_future_clear_or", "3s +clear +oráculo", semillas=3, guardar=True, feats="base,logic,past,future,clear",
       sesion=True, oraculo=True)

### 12c · Versión final con oráculo, análisis por controlador y comprobación en test

**Ejecuciones guardadas** (terminal, `--sesion --oraculo`; suavizado `prob` con CV anidada, mismo código que la celda de 9b):

| Modelo | semillas | 1.er pase | + sesión | + suavizado (CV anidada) | params suavizado (finales) | peor controlador |
|---|---|---|---|---|---|---|
| `lgb_future_clear10_ses` (envío actual, Kaggle 0.947) | 10 | 0.9086 | 0.9158 | 0.9181 | (0.85, 3) | 0.784 |
| `lgb_future_clear_or` | 3 | 0.9091 | 0.9236 | **0.9250** | (1.0, 10) | 0.769 |
| **`lgb_future_clear10_or`** | **10** | 0.9104 | 0.9215 | **0.9227** | (1.0, 5) | 0.769 |

- La diferencia 3 frente a 10 semillas (0.2) es ruido de las semillas. **Para enviar se elige la de 10 semillas**:
  sus predicciones de test son más estables. Mejora sobre el envío actual: **+0.5 en CV**.
- El suavizado suma ahora +0.12–0.14 (antes +0.23): parte de lo que aportaba ya lo da el oráculo.
- **Variante F** (oráculo dentro del logit de sesión: `or_near180` medio en la racha, `or_ever`, `or_near180`): 0.9227–0.9230
  frente a 0.9236. **No aporta**: lo que da el oráculo ya lo recoge el 1.er pase.

**Por controlador** (3 semillas + sesión, frente a `lgb_future_clear_ses`):

| atco | 001 | 016 | 008 | 010 | 004 | 002 | 005 | 017 | 007 | 015 | 006 | 003 | 018 | 020 | 019 | 011 | 009 | 014 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| sin | 0.781 | 0.856 | 0.861 | 0.811 | 0.842 | 0.932 | 0.908 | 0.910 | 0.928 | 0.927 | 0.901 | 0.949 | 0.953 | 0.910 | 0.932 | 0.956 | 0.962 | 0.971 |
| con | 0.763 | 0.859 | 0.868 | 0.869 | 0.886 | 0.905 | 0.911 | 0.920 | 0.934 | 0.936 | 0.939 | 0.940 | 0.951 | 0.952 | 0.961 | 0.967 | 0.969 | 0.979 |
| Δ | −1.8 | +0.4 | +0.7 | **+5.7** | **+4.4** | −2.7 | +0.3 | +1.0 | +0.6 | +0.9 | **+3.7** | −1.0 | −0.3 | **+4.3** | **+2.9** | +1.0 | +0.7 | +0.8 |

- **Suben 14 de 18 controladores**, sobre todo los flojos. **El 020 (leaderboard público) sube +4.3.**
- Bajan 4 (001, 002, 003, 018). El peor sigue siendo el **001** (una sola sesión, 338 snapshots): pierde 6 snapshots
  (4 entry_coordination, 2 zone_conflict). Por sesiones, bajan 14 de 37 (la que más, 002sc3: −4.1).
- **Lectura:** el oráculo empuja hacia lo que hace "el controlador típico" del escenario: ayuda mucho a la mayoría y
  quita algo a los atípicos. **Riesgo para el privado:** que el 023 sea atípico. Ganancia media +0.9 frente a una
  pérdida máxima de −2.7.

**Comprobación en test (antes de subir):** las features del oráculo en test tienen la misma distribución que en el OOF
(`or_near180`: media 0.154 frente a 0.166 y 41 % frente a 40 % > 0; `or_ever`: 0.385 frente a 0.398; vacíos 0.8 % frente
a 0.5 %), y el tiempo alineado de las 5 sesiones de test cae en el rango de train (0–3460 s). No hay indicios de fallo.

**Envíos:**
- `futuro/submissions/sub_suav_lgb_future_clear10_or_ses.csv`: **la mejor** (CV anidada 0.923).
- `futuro/submissions/sub_suav_lgb_future_clear10_ses.csv`: **la segura**, sin oráculo (0.918, Kaggle 0.947).

En test, el 46 % de los snapshots cambian su top-3 respecto al envío actual. **Cómo leer el público** (±2–3 de ruido):
≥ 0.94 → el oráculo funciona en test; muy por debajo de 0.947 (p. ej. < 0.92) → algo falla y no se usa.

### 12d · Kaggle y ganancia por escenario (21/10, 00:50)

> ⚠️ **Corregido en 14c:** sin suavizar, el oráculo mejora en los tres escenarios. Lo de "solo el escenario 2" venía
> del suavizado y del ruido de las semillas. La recomendación de entregar una versión con oráculo y otra sin él se mantiene.

**Kaggle público (controlador 020, sesiones 020sc2 y 020sc3):** `sub_suav_lgb_future_clear10_or_ses.csv` = **0.940**,
frente a 0.947 del envío sin oráculo. Son −0.7 (~6 snapshots), dentro del ruido del público (±2–3). Pero la CV del 020
esperaba +4.3, así que es una señal (≈ 2 desviaciones) de que el oráculo transfiere peor de lo que decía la CV.

**Ganancia por escenario** (OOF, ambos con 10 semillas + sesión + suavizado):

| Escenario | sesiones | sin oráculo | con oráculo | Δ media | sesiones que suben / bajan | Δ mediana por sesión |
|---|---|---|---|---|---|---|
| 1 | 14 | 0.9033 | 0.9118 | +0.9 | 6 / 6 | **0.0** |
| 2 | 12 | 0.9233 | 0.9325 | +0.9 | 8 / 4 | +1.1 |
| 3 | 11 | 0.9294 | 0.9289 | **−0.05** | 3 / 6 | **−0.5** |

- **Solo el escenario 2 mejora de forma consistente.** En el 1, la media sube por unas pocas sesiones con ganancias
  grandes (controladores 010, 004…), pero la sesión típica no cambia. En el 3 no aporta o resta.
- Encaja con el público: 020sc2 (escenario 2) + 020sc3 (escenario 3). El +4.3 del 020 en la CV venía de su sesión
  020sc1, uno de los casos que ganan mucho: era engañoso.
- **Para el privado (023: escenarios 1, 2 y 3):** ganancia esperada pequeña, positiva e incierta, con riesgo a la baja.
- En test cambia el top-3 del 40–51 % de los snapshots de cada sesión, y el top-1 del 5–12 %.

**Decisión recomendada para las 2 entregas finales:** en Kaggle la nota final es la **mejor de las 2 seleccionadas**,
así que se elige **una sin oráculo y otra con oráculo** (cobertura gratis):
1. `sub_suav_lgb_future_clear10_ses.csv`: sin oráculo, CV 0.918, público 0.947.
2. `sub_suav_lgb_future_clear10_or_ses.csv`: con oráculo, CV 0.923, público 0.940.

**Lección:** con folds por controlador, la media de la CV la mueven unas pocas sesiones. Para decidir hay que mirar
también la **mediana por sesión** y **por escenario**, no solo el peor controlador.

## 13 · Tanda f5 de Juanma: `extra_trees`, árboles grandes, red con `clear` y stacks con sesión (20/10 23:00 – 21/10 00:30)

Desde la terminal con `futuro/pruebas/f5_clear_et.py` (variantes del 1.er pase con `base_fut.lgb_cv`: mismos parámetros,
features y folds que `m_lgb_future.py`) y `futuro/pruebas/f5_stack.py` (modos `ses` = segundo paso de `feat_session.sesion`
sobre cualquier oof/preds, `stack` = `stacking/stack.py` y `suav` = suavizado `prob` con CV anidada). Salidas en
`futuro/pruebas/oof|preds|submissions` y registro en `futuro/pruebas/resultados.jsonl` (hilos `f5_clear_et`, `f5_stack`).
Todo `+ clear` (base + logic + past + future + clear). Control: `f5_control_clear` reproduce `lgb_future_clear` (0.9071).

**13a · 1.er pase y segundo paso de sesión**

| Modelo | semillas | 1.er pase | + sesión | + suavizado (CV anidada) | peor controlador |
|---|---|---|---|---|---|
| `lgb_future_clear10` (referencia, 9b) | 10 | 0.9086 | 0.9158 | 0.9181 | 0.784 |
| `f5_clear_et` (`extra_trees`) | 3 | 0.9081 | 0.9161 | — | 0.787 |
| `f5_clear_et10` | 10 | 0.9077 | 0.9147 | 0.9157 | 0.778 / 0.790 |
| `f5_clear_l63_noet` (63 hojas, `min_child=100`) | 3 | 0.9077 | 0.9159 | — | 0.766 |
| `f5_clear_l127` (127 hojas + `extra_trees`) | 3 | 0.9076 | 0.9160 | — | 0.781 |
| `f5_clear_l63` (63 hojas, `min_child=100`, `extra_trees`) | 3 | 0.9106 | 0.9196 | 0.9207 | 0.778 |
| **`f5_clear_l63_10`** | **10** | **0.9113** | **0.9184** | **0.9202** | 0.778 / 0.769 |
| `f5_clear_binary` / `f5_clear_xendcg` | 3 | 0.9044 / 0.8978 | 0.8713 / 0.8934 | — | 0.680 / 0.710 |

- Solo gana la **combinación** 63 hojas + `extra_trees`; cada cosa por separado se queda en ~0.916 con sesión (igual que 12a).
- `binary` y `xendcg` empeoran mucho con el segundo paso (escala de scores distinta a la de LambdaRank).

**13b · Red con futuro + `clear` (lo pendiente de la sección 10)**: `m_nn_future.py --clear` (opción nueva, añade `sx_*`).
`nn_future_clear` / `_d128` / `_bs256` / `_h8` + sesión: 0.872–0.882 (la red sin `clear` + sesión: 0.878; sin sesión 0.887).
**El segundo paso empeora a la red** y en todos los stacks recibe peso ~0. Descartada.

**13c · Stacks** (media ponderada de `stack.py`, CV anidada)

| Stack | CV | + suavizado (anidada) | peor controlador |
|---|---|---|---|
| clear, f1_et, nn, linear, f4 (sin sesión) | 0.9085 | 0.9104 | 0.772 |
| sesión a cada modelo → stack (clear10, et10, redes, linear, f4) | 0.9186 | 0.9169 | 0.784 |
| stack → sesión (los mismos sin sesión) | 0.9164 | 0.9189 | 0.784 |
| 11 modelos con sesión (+ binary, xendcg, l63, redes) | 0.9179 | 0.9197 | 0.760 |
| `l63_10_ses` + `clear10_ses` + `linear_ses` + `f4_ses` | 0.9195 | 0.9197 | 0.772 |

Ningún stack mejora a `suav_f5_clear_l63_10_ses` solo; los dos órdenes (sesión antes o después del stack) quedan iguales.

**13d · Por escenario y sesión (criterio de 12d)**, `suav_f5_clear_l63_10_ses` frente a `suav_lgb_future_clear10_ses`:

| Escenario | Δ media | Δ mediana por sesión | sesiones que suben / bajan |
|---|---|---|---|
| 1 | −0.24 | −0.37 | 4 / 8 |
| 2 | **+1.26** | **+1.26** | 8 / 3 |
| 3 | −0.12 | −0.51 | 4 / 6 |

Total: 16 sesiones suben y 17 bajan, mediana 0. Controlador 020 (público): +3.8 en CV.
**Mismo patrón que el oráculo (12d):** el +0.25 de la media sale del escenario 2; en 1 y 3 resta un poco. No es robusto.


In [ ]:
# Tabla de la tanda f5 desde el registro (no reentrena nada)
import json
_r = {}
for l in open(ROOT / "futuro" / "pruebas" / "resultados.jsonl"):
    x = json.loads(l)
    if x["hilo"] in ("f5_clear_et", "f5_stack"):
        _r[x["nombre"]] = x
pd.DataFrame([dict(nombre=n, cv=x["cv"], cv_anidada=x.get("cv_anidada"), peor_controlador=x["min_atco"],
                   std_folds=x["std_folds"], preds=x["tiene_preds"]) for n, x in _r.items()]) \
    .sort_values("cv", ascending=False).reset_index(drop=True)


In [ ]:
# Reproducir (terminal, desde la raíz; ~15 min con 28 hilos):
#   python futuro/pruebas/f5_clear_et.py f5_clear_l63_10
#   python futuro/pruebas/f5_stack.py ses f5_clear_l63_10
#   python futuro/pruebas/f5_stack.py suav f5_clear_l63_10_ses
# Envío: futuro/pruebas/submissions/sub_suav_f5_clear_l63_10_ses.csv


## 14 · Lectura de la tanda f5 de Juanma, criterio por controlador y oráculo de coordinaciones (21/10, 00:55–01:30)

**14a · Qué deja la sección 13 (Juanma)**
- `f5_clear_l63_10` (63 hojas + `min_child=100` + `extra_trees`, 10 semillas) + sesión + suavizado = **0.9202** (CV
  anidada), frente a 0.9181. Por separado, los cambios no ganan (lo mismo que en 12a).
- La red con `clear` y los stacks no mejoran al LightGBM solo (igual que en 5 y 6c).
- Mismo patrón por escenario que el oráculo en 12d. **Además, los dos ganan y pierden en los mismos controladores**
  (ganan 006, 004, 020, 010; pierden 002, 001, 003, 018): son el mismo tipo de cambio, más seguros en los casos típicos.

**14b · Nuevo criterio: ganancia por controlador** (`futuro/pruebas/comparar.py`, reutilizable por todo el equipo)

El privado es un controlador nuevo. `comparar.py <ref> <cand>` da la CV, la ganancia por controlador (media = esperanza
para un controlador nuevo, mediana, suben/bajan), un bootstrap sobre los 18 controladores (IC 90 %, P(media > 0)) y la
ganancia por escenario.

**Limitación importante:** el conjunto de semillas mueve a **todos** los controladores a la vez, y el bootstrap por
controladores no lo ve. Ejemplo: `lgb_future_clear10_or_ses` frente a `lgb_future_clear_or_ses` (mismo modelo, 10
frente a 3 semillas) = −0.21, bajan 12 de 18, IC [−0.46, −0.12]. Diferencias de ±0.2 entre versiones son ruido de
semillas aunque parezcan "significativas".

**14c · Oráculo: corrección de 12d y variantes con 3 semillas** (todas frente a `lgb_future_clear_ses`, 3 semillas,
sin suavizar)

| Modelo (3 semillas + sesión) | CV | peor controlador | por controlador: media / mediana | suben / bajan | IC 90 % | escenarios 1 / 2 / 3 |
|---|---|---|---|---|---|---|
| oráculo (`lgb_future_clear_or_ses`) | **0.9236** | 0.763 | **+1.21** / +0.77 | 14 / 4 | [+0.40, +2.07] | +1.2 / +0.8 / +0.9 |
| **oráculo solo en coordinaciones** (`--oraculo-var coord`) | 0.9220 | **0.784** | +0.94 / +0.74 | 14 / 4 | [+0.36, +1.50] | +0.4 / +1.3 / +0.6 |
| oráculo + 63 hojas + `extra_trees` (`--extra-trees`) | 0.9188 | 0.787 | +0.86 / +0.28 | 9 / 8 | [+0.03, +1.74] | +0.7 / +0.9 / −0.3 |
| 10 semillas: oráculo frente a sin oráculo | 0.9215 frente a 0.9158 | 0.763 | +0.86 / +0.38 | 10 / 8 | [+0.01, +1.76] | +0.8 / +0.6 / +0.3 |

- **Corrección de 12d:** sin suavizar, el oráculo mejora **en los tres escenarios** (también en el 3). El "0.0 en el
  escenario 3" de 12d venía del suavizado y del ruido. La ganancia es más robusta de lo que se dijo allí.
- **Oráculo + 63 hojas + `extra_trees` no se suman** (0.9188 < 0.9236): corrigen los mismos casos.
- **Oráculo de coordinaciones frente a completo** (directo): −0.27 de media, IC [−0.89, +0.35], suben 6 / bajan 12.
  El completo gana algo en media; el de coordinaciones **protege al peor controlador** (0.784 frente a 0.763), porque
  no toca zone_conflict ni los conflictos, donde el oráculo restaba. Es un equilibrio entre ganancia y riesgo.
- **La alineación no explica lo del escenario 3:** allí es la mejor (desviación mediana de 0 s frente a 9 s y 2.5 s), y
  el acuerdo entre controladores es igual en los tres (53–58 % trabajan la misma tarea a ±180 s; 24–26 % en coordinaciones).

In [ ]:
# Comparaciones de esta sección (no reentrena nada; ~10 s cada una)
sys.path.insert(0, str(ROOT / "futuro" / "pruebas"))
from comparar import comparar
for c in ["lgb_future_clear_or_ses", "lgb_future_clear_orcoord_ses", "lgb_future_clear_or_l63et_ses"]:
    comparar("lgb_future_clear_ses", c)
comparar("lgb_future_clear_or_ses", "lgb_future_clear_orcoord_ses")

### 14d · Oráculo de coordinaciones con 10 semillas: nuevo mejor modelo (21/10, 01:15)

`python futuro/m_lgb_future.py --feats base,logic,past,future,clear --sesion --oraculo --oraculo-var coord --seeds 10 --name lgb_future_clear10_orcoord`

Todas las métricas de la regla de la cabecera, frente a `lgb_future_clear10_ses` (10 semillas, sin oráculo, sin suavizar):

| | CV | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | Δ por controlador (media / mediana) | IC 90 % | peor controlador | escenarios 1 / 2 / 3 |
|---|---|---|---|---|---|---|---|---|---|
| `lgb_future_clear10_ses` (referencia) | 0.9158 | — | — | — | — | — | — | 0.781 (001) | — |
| oráculo completo (`lgb_future_clear10_or_ses`) | 0.9215 | +0.57 | +0.00 | 18 / 17 | 10 / 8 | +0.86 / +0.38 | [+0.01, +1.76] | 0.763 (001) | 0.908 / 0.927 / 0.930 |
| **oráculo de coordinaciones (`lgb_future_clear10_orcoord_ses`)** | **0.9244** | **+0.86** | **+0.99** | **25 / 9** | **14 / 4** | **+1.13 / +1.06** | **[+0.47, +1.80]** | **0.790 (001)** | 0.905 / 0.936 / 0.933 |

- **Cumple la regla de adopción:** mejora la media **y** la mediana por sesión, sube en 25 sesiones y baja en 9, y el peor
  controlador también mejora.
- Con 10 semillas, el oráculo completo deja la mediana por sesión igual (18 / 17): era más frágil de lo que parecía con 3.
  Directo, coordinaciones frente a completo: +0.29, controladores 11 / 7.
- **Suavizado `prob` encima** (CV anidada): 0.9244 → **0.9251** (+0.07); parámetros (1.0, 2) en 4 de 5 folds; peor
  controlador 0.796; frente a `suav_lgb_future_clear10_ses`: sesiones 23 / 9, Δ mediana por sesión +0.57.
- Para comparar: `suav_f5_clear_l63_10_ses` (Juanma) frente a la misma referencia: +0.26, mediana por sesión +0.00, 16 / 17.

**Entregas finales recomendadas** (Kaggle se queda con la mejor de las dos seleccionadas):
1. **`futuro/submissions/sub_suav_lgb_future_clear10_orcoord_ses.csv`**: la mejor (CV anidada 0.925, peor controlador 0.796).
2. `futuro/submissions/sub_suav_lgb_future_clear10_ses.csv`: la segura, sin oráculo (CV 0.918, público 0.947).

### 14e · Kaggle: el oráculo de coordinaciones saca 0.934 en el público (21/10, 01:25)

| Envío | CV | público (020: sesiones 020sc2 y 020sc3) |
|---|---|---|
| sin oráculo (`sub_suav_lgb_future_clear10_ses`) | 0.918 | **0.947** |
| oráculo completo (`sub_suav_lgb_future_clear10_or_ses`) | 0.923 | 0.940 |
| oráculo de coordinaciones (`sub_suav_lgb_future_clear10_orcoord_ses`) | **0.925** | 0.934 |

**¿Qué probabilidad tiene −1.3 según la CV?** Δ por sesión (con oráculo de coordinaciones − sin oráculo, ambos 10
semillas + sesión + suavizado), en todas las parejas (sesión del escenario 2, sesión del escenario 3), como las del público:
media +0.98; **P(Δ medio ≤ −1.3) = 0.08**; P(Δ ≤ 0) = 0.21.

Δ por sesión en la CV: de −5.5 (002sc3) a +7.1 (004sc1). Hay controladores en los que el oráculo hace daño (002: −5.5
y −2.0; 018sc1: −3.7). El 020 ganaba +3.0 en su sesión de CV (020sc1).

**Lectura:** evidencia moderada de que en el 020 el oráculo no ayuda (dos envíos por debajo; el último, en el 8 % de
la cola). **No** indica un fallo en test (entonces caería muy por debajo de 0.92), y la CV completa sigue en +0.9.

**Decisión:** se refuerza la cobertura. Las 2 entregas finales son **una con oráculo y otra sin él** (Kaggle se queda
con la mejor):
1. `sub_suav_lgb_future_clear10_orcoord_ses.csv`: mejor CV con todos los criterios (14d).
2. `sub_suav_lgb_future_clear10_ses.csv`: sin oráculo, la mejor en el público.
No elegir dos versiones con oráculo.

### 14f · Diagnóstico del oráculo en el controlador 020 (21/10, 01:50; sin reentrenar)

**¿Hay un fallo técnico en las sesiones del 020?** No.
- **Alineación:** corrección 0 s y dispersión 0 s en 020sc2 y 020sc3. En train/val, la mediana es 0 s; el caso más raro es
  016sc1 (−103 s). En test, **023sc1 tiene −47 s** (dentro de lo visto en train).
- **Valores del oráculo en sus coordinaciones:** `or_near180` > 0 en el 21 % (020sc2) y el 17 % (020sc3), frente a cuartiles
  del 18–29 % en las sesiones etiquetadas; 023: 17–18 %. Normal.

**¿Qué hace distinto el oráculo en el 020?** (test, `sub_suav_lgb_future_clear10_orcoord_ses` frente a
`sub_suav_lgb_future_clear10_ses`)

| Sesión | snapshots con top-3 distinto | entry_coordination: salen / entran | exit_coordination: salen / entran | top-1 = coordinación (sin → con oráculo) |
|---|---|---|---|---|
| 020sc2 | 51 % | 133 / 53 | 77 / 53 | 9.1 % → 9.3 % |
| 020sc3 | 49 % | 118 / 25 | 60 / 39 | 11.1 % → 10.4 % |
| 023sc1 | 33 % | 19 / 12 | 47 / 30 | 3.3 % → 3.3 % |
| 023sc2 | 38 % | 49 / 8 | 41 / 22 | 5.2 % → 5.2 % |
| 023sc3 | 38 % | 53 / 0 | 52 / 62 | 3.4 % → 1.5 % |

- En el 020 el oráculo **cambia más** y, sobre todo, **saca entry_coordination del top-3**.
- **El 020 es muy de coordinaciones:** el 21 % de sus respuestas en 020sc1 (mediana 13 %, rango 7–26 %). El indicador sin
  etiquetas "top-1 del modelo sin oráculo = coordinación" correlaciona 0.80 con la fracción real por sesión, y le da 9–11 %
  en test frente al 3–5 % del 023.

**¿Castiga el oráculo a los controladores con muchas coordinaciones?** La CV **no** lo apoya: Spearman(fracción de
coordinaciones, ganancia del oráculo) = +0.18 por controlador (p = 0.47) y −0.10 por sesión. Por terciles de controladores,
los de muchas coordinaciones ganan **más** (+1.68, frente a +1.07 y +0.2).

**Conclusión:** no se puede atribuir el 0.934 del público a ese mecanismo. Lo más defendible es ruido más algo propio del
020. Para el 023 (pocas coordinaciones según el indicador), el oráculo cambia menos predicciones y pesa menos, para bien o
para mal. **Las entregas finales no cambian:** `sub_suav_lgb_future_clear10_orcoord_ses.csv` +
`sub_suav_lgb_future_clear10_ses.csv`.

### 14g · Selector de modelos por escenario con el oráculo de coordinaciones como candidato (21/10, 02:00; sin reentrenar)

**Ya probado por Juanma** (sección 15, `f6_ideas.py escen`): selección **anidada** por escenario (elige en 4 folds y mide
en el 5.º) entre A = `suav_lgb_future_clear10_ses`, L = `suav_f5_clear_l63_10_ses` y O = `suav_lgb_future_clear10_or_ses`
→ 0.9226, sesiones 13 / 13, frágil. Lo único nuevo aquí es añadir **C = `suav_lgb_future_clear10_orcoord_ses`** como
candidato (mismo procedimiento, sobre los mismos OOF suavizados).

| Candidatos | CV anidada | frente a C solo | peor controlador | sesiones frente a A (suben / bajan) | elegidos por escenario (folds 0–4) |
|---|---|---|---|---|---|
| A + L + O (Juanma) | 0.9226 | −0.33 | 0.769 | 13 / 13 | 1: OOOOO · 2: LLLOL · 3: OAAAO |
| A + L + C | 0.9229 | −0.30 | 0.784 | 20 / 9 | 1: CACCC · 2: CCCCL · 3: CCCCC |
| A + L + O + C | 0.9239 | −0.20 | 0.769 | 19 / 14 | 1: OOOOO · 2: CCCOL · 3: CCCCC |
| A + C | 0.9241 | −0.18 | 0.784 | 21 / 8 | 1: CACCC · 2: CCCCC · 3: CCCCC |
| C + L | 0.9247 | −0.12 | 0.796 | 22 / 10 | 1: CCCCC · 2: CCCCL · 3: CCCCC |
| **C solo** | **0.9259** (misma escala; 0.9251 en CV anidada) | — | **0.796** | 23 / 9 | — |

**Conclusión:** con C entre los candidatos, el selector lo elige casi siempre y en los tres escenarios. Elegir por
escenario solo añade decisiones que cambian entre folds y resta (−0.12 a −0.33). La ventaja de L en el escenario 2 y la
debilidad de O en el 3 no son sistemáticas. **Se queda C solo** (`orcoord`, 14d).

## 15 · Otros ámbitos: secuencia, controlador nuevo, etiquetas ruidosas, tramo y stacks con oráculo (Juanma, 21/10, 00:40–01:45)

Scripts: `futuro/pruebas/f6_ideas.py` (post-procesos sin reentrenar: `hmm`, `prior`, `escen`), `futuro/pruebas/f6_soft.py`
(1.er pase con etiquetas ruidosas), `futuro/pruebas/f7_tramo.py` (modelo por racha) y `f5_stack.py` (stacks). Registro en
`futuro/pruebas/resultados.jsonl`, hilos `f6`, `f7`, `f5_stack`. Referencia A = `suav_lgb_future_clear10_ses` (0.9181).

| Idea | Qué hace | Resultado | Veredicto |
|---|---|---|---|
| **HMM en la racha** (`hmm`) | cadena oculta: sigue con la tarea con prob. *s* o cambia; emisión p^τ; forward-backward; *s*, τ con CV anidada | sobre `clear10_ses` 0.9172; sobre `l63_10_ses` 0.9207 (suavizado: 0.9181 / 0.9202) | igual que el suavizado |
| **Adaptación al controlador** (`prior`, EM de Saerens) | reescala p(tarea) por (π_controlador / π_resto)^λ del tipo, estimado sin etiquetas | λ = 0 en los 5 folds | sin señal |
| **Etiquetas ruidosas** (`f6_soft.py`) | relevancia 2 = respuesta, 1 = respuesta a ±k snapshots (`grado1/3`); o quitar respuestas aisladas | `grado3` 0.9092 → 0.9151 con sesión; `grado1` 0.9133; `aislada` 0.9119 (control 0.9071 → 0.9142) | ruido |
| **Modelo por escenario** (`escen`) | por escenario, el mejor de A / l63 / oráculo en los folds de entrenamiento | 0.9226, mediana por sesión 0, sesiones 13 / 13 | frágil |
| **Modelo por racha** (`f7_tramo.py`) | LambdaRank sobre (racha, tarea) con agregados del 1.er pase y de `future`/`clear`; score = lsm + w · racha | solo racha 0.898; combinado 0.9064 (1.er pase 0.9086); + sesión 0.9147 (0.9158) | no aporta |
| **Stack con oráculo** (`stackf_or`) | `clear10_or_ses` 0.47, `l63_10_ses` 0.12, `linear_ses` 0.18, `f4_ses` 0.24; + suavizado | **0.9256** (anidada), peor controlador 0.781, sesiones 23 / 7, controladores 14 / 4 | empata con `orcoord` (14d) |
| Stack con oráculo de coordinaciones (`stackf_orc`, `stackf_orc2`) | lo mismo con `clear10_orcoord_ses` | 0.9204 / 0.9214 (`orcoord` solo: 0.9244) | **peor que el modelo solo** |

**Análisis de errores** (`suav_f5_clear_l63_10_ses`): la correcta está en top-3 el 92.1 %, en 4–5 el 2.9 %, en 6–10 el
3.4 %, más allá el 1.6 %. La mitad de los fallos son coordinaciones (exit 28.5 %, entry 22.8 %) y en esos casos el top-1
suele ser otra coordinación. Las rachas se aciertan o se fallan enteras (92.6 % todo bien, 3.8 % todo mal, 3.5 % mixtas):
por eso ningún post-proceso en la racha (suavizado, HMM, modelo por racha) arregla lo que queda.

**Conclusiones:**
- Con la información del snapshot, la sesión y la racha, los post-procesos están saturados (todos ±0.002).
- La única fuente nueva que suma es el oráculo (otros controladores). Stack con oráculo completo y `orcoord` solo empatan
  (`comparar.py`: IC 90 % [+0.27, +1.7] en los dos). Se prefiere **`orcoord`**: mejor peor controlador (0.796 frente a
  0.781) y un solo modelo; los pesos del stack no son estables (con `orcoord` dentro, el stack queda por debajo).


## 16 · Atacar las coordinaciones desde el modelo: peso, especialista y features relativas (21/10, 02:05–02:30)

Base: `lgb_future_clear_orcoord` (3 semillas; 1.er pase 0.9101, + sesión **0.9220**, peor controlador 0.784).
Todo con 3 semillas y medido contra esa base. Las ideas 4 (suavizado adaptativo) y 5 (modelo por racha) no se repiten:
equivalen al HMM y a `f7_tramo.py` de la sección 15.

| Prueba | Qué cambia | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | peor controlador |
|---|---|---|---|---|---|---|---|---|
| peso ×1.5 (`--peso-coord 1.5`) | los snapshots cuya respuesta es una coordinación pesan 1.5 en el ranking | 0.9101 | 0.9211 | −0.09 | +0.00 | 15 / 14 | 6 / 9 | 0.778 |
| peso ×2.0 (`--peso-coord 2.0`) | ídem, 2.0 | 0.9093 | 0.9233 | +0.13 | +0.20 | 20 / 10 | 10 / 7 | 0.775 |
| bloque `rel` (`futuro/feat_rel.py`) | diferencia con el mejor competidor del mismo tipo / snapshot (siguiente instrucción, desaparición, `sx_task`), rango dentro del tipo, nº de tareas del avión que encajan con la siguiente instrucción, nº de aviones con instrucción en ≤ 30 s | 0.9094 | 0.9236 | +0.17 | +0.00 | 17 / 14 | 9 / 7 | 0.775 |
| especialista (`futuro/pruebas/especialista.py`) | ranker solo con candidatos de coordinación; base + β · especialista en esas filas, β en CV anidada | — | 0.9211 (anidada) | −0.09 | — | — | — | — |

- **Peso:** ×1.5 y ×2.0 se mueven en sentidos opuestos (−0.09 / +0.13), y la media por controlador no cambia: ruido de semillas.
- **`rel`:** la señal existe (la correcta tiene la siguiente instrucción 34 s antes que su mejor competidor del mismo
  tipo; el resto, 167 s después), pero el modelo ya la sacaba de los rangos que había. +0.17 con la mediana por sesión
  igual y el peor controlador peor: no se adopta.
- **Especialista:** la CV anidada elige β = 0 en 4 de 5 folds; con β > 0, cuanto más pesa, peor (0.25 → 0.9212; 0.5 →
  0.9113; 1 → 0.8908; 2 → 0.8437). Entrenado solo con coordinaciones, no sabe cuándo la respuesta NO es una coordinación.

**Conclusión:** ninguna de las tres cumple la regla de adopción. El modelo no saca más de las coordinaciones con la
información disponible; lo único que ha sumado en ellas es el oráculo. **Se queda `orcoord` (14d).**

*(Nota de código: desde aquí `--ref` es el nombre del 1.er pase de referencia; el 1.er pase se compara con `<ref>` y el
segundo paso con `<ref>_ses`. Antes se comparaba el 1.er pase con `<ref>_ses`; esas líneas del registro no son válidas.)*

## 17 · Oráculo de coordinaciones ponderado por estilo y auditoría de ambigüedad (21/10)

Base: `lgb_future_clear_orcoord` (3 semillas; + sesión **0.9220**, peor controlador 0.784). Script de los diagnósticos:
`futuro/pruebas/auditoria_coord.py` (sin reentrenar). El modelo de eventos temporales (hazard) **no se probó**: el segundo
paso de sesión ya modela que cada tarea se trabaja una vez y el suavizado modela las rachas.

**a) ¿Los controladores de estilo parecido coinciden más?** Perfil de estilo por sesión **sin etiquetas**
(`feat_oracle.estilo`, z-scores): cuándo asume y transfiere respecto al borde del sector, ritmo de instrucciones y mezcla
de tipos (nivel, lateral, rumbo, asunción forzada). En 424 parejas de sesiones del mismo escenario y distinto controlador,
Spearman(distancia de estilo, acuerdo de respuestas a ±180 s) = **−0.20** (p = 2e-5). El acuerdo es 0.575 con el tercio de
referencias más parecidas, frente a 0.529 con el menos parecido. La señal es real, pero pequeña.
- **023 (privado) es atípico:** 023sc2 es más atípica que cualquier sesión etiquetada (percentil 1.00; `frac_forzado`
  z = 3.51, `tts_al_asumir` z = 2.48) y 023sc1 está en el percentil 0.92 (`tts_al_asumir` z = 3.55). 020 es típico
  (percentiles 0.14 y 0.19).

**b) Oráculo ponderado** (`--oraculo-var coord --oraculo-temp T`): cada sesión fuente pesa exp(−distancia de estilo / T)
en vez de pesar todas igual; T = ∞ es el uniforme.

| T | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | peor controlador | P(ganancia media > 0), bootstrap |
|---|---|---|---|---|---|---|---|---|
| ∞ (uniforme) | 0.9101 | 0.9220 | — | — | — | — | 0.784 (001) | — |
| 2 | 0.9120 | 0.9230 | +0.10 | +0.00 | 15 / 15 | 9 / 6 | 0.790 (001) | 0.52 |
| 1 | 0.9113 | 0.9215 | −0.05 | +0.00 | 14 / 15 | 8 / 7 | 0.743 (001) | 0.14 |
| 0.5 | 0.9094 | 0.9218 | −0.02 | +0.00 | 16 / 16 | 5 / 11 | 0.781 (001) | 0.21 |

- **CV anidada** (elige T en 4 folds y mide en el 5.º, entre ∞, 2, 1 y 0.5): T = 2 en los 5 folds → **0.9230**. Son +0.10
  sobre el uniforme, dentro del ruido de semillas (±0.2, sección 14b), con la mediana por sesión sin cambios y la ganancia
  media por controlador en +0.00.
- **¿Ayuda a las sesiones atípicas, que es lo que importaría para 023?** No. Spearman(atipicidad, Δ por sesión) = +0.12
  (p = 0.48) con T = 2, y −0.01 / −0.06 con T = 1 / 0.5. En las 8 sesiones más atípicas, Δ = +0.36 con T = 2 (resto +0.04),
  pero con T más bajo, que pondera más, el efecto desaparece en vez de crecer.

**c) Auditoría de ambigüedad de las coordinaciones.** Snapshots cuya respuesta es una coordinación, mirando solo las
candidatas de su mismo tipo. Se compara el modelo (`lgb_future_clear_orcoord`) con un kNN (25 vecinos, por folds) sobre las
mismas features.

| | snapshots | top-1 al azar | top-1 modelo | top-1 kNN | AUC por parejas modelo | AUC kNN | vecino más parecido con etiqueta contraria (10 % más cercanos) |
|---|---|---|---|---|---|---|---|
| entry_coordination | 1068 | 0.105 | **0.727** | 0.546 | **0.950** | 0.863 | 20 % (n = 10) |
| exit_coordination | 736 | 0.225 | **0.701** | 0.433 | **0.851** | 0.691 | 42 % (n = 52) |

- El modelo ya separa mucho mejor que los vecinos: estos no traen señal que el modelo no use.
- En exit_coordination, el 42 % de los contextos casi idénticos de otro controlador tienen la etiqueta contraria. Hay
  un **techo de información**: qué coordinación concreta se hace depende del controlador, no de lo observable.

**Conclusión:** no se adopta el oráculo ponderado; se queda `orcoord` uniforme. La similitud de estilo existe pero es
demasiado débil para mejorar el promedio, y 023 está fuera del rango de estilos vistos, así que ponderar por parecido
tampoco tiene referencias cercanas en las que apoyarse. La auditoría explica por qué las secciones 11 y 16 no lograron
subir las coordinaciones: la ambigüedad que queda está en las etiquetas, no en el modelo. No se gastan 10 semillas en T = 2.

In [ ]:
# Reproducir (terminal, desde la raíz):
#   for T in 2 1 0.5; do python futuro/m_lgb_future.py --feats base,logic,past,future,clear --sesion --oraculo \
#       --oraculo-var coord --oraculo-temp $T --threads 10 --ref lgb_future_clear_orcoord --name lgb_future_clear_orcoord_t${T/./}; done
#   python futuro/pruebas/auditoria_coord.py          # a) estilo, b) atipicidad frente a Δ, c) ambigüedad (~3 min)
# Comparaciones (no reentrena nada):
from comparar import comparar
for t in ("t2", "t1", "t05"):
    comparar("lgb_future_clear_orcoord_ses", f"lgb_future_clear_orcoord_{t}_ses")

## 18 · Información nueva: geometría del tráfico y adaptación al controlador por covariate shift (21/10)

Referencia: `lgb_future_clear` **sin oráculo** (3 semillas, mismos folds y semillas; + sesión **0.9142**, peor controlador
0.781). Las columnas `entry` y `exit` dan el acierto top-3 en los snapshots cuya respuesta es entry_coordination o
exit_coordination.

**a) Geometría** (`futuro/feat_geo.py`, bloques `geoA`, `geoB`, `geoC`). Los datos **no traen posiciones** (ni lat/lon ni
rumbo), así que no hay CPA/TCPA entre parejas de aviones. Con lo que hay por avión:
- **A · borde del sector:** velocidad real de acercamiento al borde en los últimos ~30 s, tiempo hasta cruzar a esa
  velocidad y orden de entrada y salida entre los aviones del snapshot.
- **B · tráfico que cruza el mismo borde a la vez:** nº de otros aviones que entran o salen a ±60/120 s, cuántos a ≤ 10/20 FL
  y mínima diferencia de tiempo y de nivel con ellos.
- **C · niveles:** velocidad vertical, FL que faltan hasta el de salida y ritmo necesario, nivel autorizado frente al de
  salida, y segundos desde el último cambio del nivel de salida y hasta el siguiente.

| Bloques | + sesión | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | peor controlador | entry | exit |
|---|---|---|---|---|---|---|---|---|
| referencia | 0.9142 | — | — | — | — | 0.781 (001) | 0.754 | 0.611 |
| A | 0.9139 | −0.02 | +0.00 | 17 / 16 | 9 / 8 | 0.766 (001) | 0.744 | 0.628 |
| B | 0.9103 | −0.39 | +0.00 | 14 / 18 | 5 / 11 | 0.775 (001) | 0.740 | 0.611 |
| C | 0.9132 | −0.10 | +0.00 | 16 / 15 | 8 / 8 | 0.784 (001) | 0.739 | 0.630 |
| A + B | 0.9120 | −0.21 | +0.00 | 16 / 16 | 5 / 12 | 0.775 (001) | 0.751 | 0.605 |
| A + B + C | 0.9126 | −0.15 | −0.24 | 13 / 19 | 8 / 8 | 0.766 (001) | 0.741 | 0.626 |

- Ningún bloque mejora. A y C suben exit_coordination (+1.7 / +1.9) pero bajan entry_coordination en una cantidad parecida.
- B, la interacción física, es el que más empeora: con ±120 s y ±10 FL marca sobre todo la densidad de tráfico, que ya
  está en `seq_len`, `l_n_*` y `logic_n_window`.

**b) Covariate shift** (`futuro/pruebas/covshift.py`). Para cada controlador de validación, un clasificador de dominio
(solo variables de entrada) estima qué snapshots de entrenamiento se parecen a ese controlador. El ranker se reentrena con
pesos w^α, con w recortado a [0.1, 10]; α = 0 es la referencia. Es nuevo frente a la sección 15: aquello reescalaba
P(tipo) y esto repondera P(X).

| α | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | peor controlador |
|---|---|---|---|---|---|---|---|
| 0 (referencia) | 0.9071 | 0.9142 | — | — | — | — | 0.781 |
| 0.5 | 0.9053 | 0.9110 | −0.32 | −0.25 | 8 / 21 | 6 / 11 | 0.778 |
| 1 | 0.9039 | 0.9131 | −0.11 | +0.20 | 19 / 15 | 9 / 9 | 0.766 |

- **CV anidada:** elige α = 0 en 4 de los 5 folds → 0.9127 (−0.15).
- **Por qué no puede funcionar:** con las filas del controlador dentro del ajuste, el clasificador lo separa «perfectamente»
  (AUC 0.999–1.000) y los pesos degeneran (tamaño efectivo 17–23 %). Pero medido **fuera de muestra**, en la otra mitad
  temporal de las sesiones del controlador, el AUC es 0.55 en el 020 y 0.69 en el 001 (0.48–0.56 con un clasificador más
  débil). Lo que distingue a un controlador son rasgos de la sesión (`blk_len`, `seq_len`, `a_n300`), no un dominio
  estable de P(X).
- Igual que la auditoría de la sección 17: entre controladores casi no cambia lo observable; cambia la decisión ante el
  mismo contexto, y eso no se corrige reponderando.

**c) Beam search sobre los OOF: no se prueba.** Una búsqueda en haz da el camino más probable de la sesión, pero el top-3
se evalúa snapshot a snapshot y necesita la probabilidad de cada tarea en cada snapshot. Eso lo calcula exactamente
forward-backward, que es el HMM de la sección 15 (sin mejora); el segundo paso de sesión ya penaliza las repeticiones.

**Conclusión:** no se adopta nada. La geometría disponible ya está en las features (los datos no traen posiciones) y no
hay un cambio de distribución estable entre controladores que aprovechar. Las entregas no cambian.

In [ ]:
# Reproducir (terminal, desde la raíz; ~6 min cada una, se pueden lanzar a la vez con --threads 6):
#   for v in A B C AB ABC; do g=$(echo $v | sed 's/./,geo&/g'); python futuro/m_lgb_future.py \
#       --feats base,logic,past,future,clear$g --sesion --threads 6 --name lgb_future_clear_geo$v; done
#   python futuro/pruebas/covshift.py 0.5 1 --threads 15     # ~17 min
from comparar import comparar
for c in ["lgb_future_clear_geoA_ses", "lgb_future_clear_geoB_ses", "lgb_future_clear_geoC_ses",
          "lgb_future_clear_geoAB_ses", "lgb_future_clear_geoABC_ses", "covshift_a0.5_ses", "covshift_a1_ses"]:
    comparar("lgb_future_clear_ses", c)

## 19 · ¿Está el problema en los datos? Ablación, análisis de errores y test frente a train (21/10)

**a) Ablación con 3 semillas sobre el modelo actual** (`--drop` nuevo en `m_lgb_future.py`). Referencia
`lgb_future_clear_ses`: 0.9142, mediana por sesión 0.925, peor controlador 0.781.

| Quita | + sesión | Δ | Δ mediana por sesión | sesiones suben / bajan | controladores suben / bajan | peor controlador | P(ganancia > 0) |
|---|---|---|---|---|---|---|---|
| `logic` (14 reglas) | 0.9147 | +0.05 | +0.00 | 14 / 17 | 9 / 8 | 0.772 | 0.52 |
| `past` (11) | 0.9140 | −0.02 | +0.00 | 17 / 15 | 8 / 8 | 0.766 | 0.34 |
| `future` (12) | 0.8649 | **−4.93** | −4.44 | 4 / 33 | 1 / 17 | 0.704 | 0.00 |
| `clear` (10) | 0.9110 | −0.32 | −0.24 | 12 / 21 | 7 / 11 | 0.775 | 0.08 |
| variables de sesión (`seq_len`, `blk_len`, `blk_pos`, `gap_*`, `x_state_age`, `a_n60`, `a_n300`) | 0.9144 | +0.02 | +0.24 | 20 / 13 | 11 / 7 | 0.769 | 0.37 |
| ritmo del controlador (`a_n60`, `a_n300`, `a_since_any`) | 0.9131 | −0.11 | +0.00 | 14 / 17 | 5 / 8 | 0.796 | 0.17 |
| **modelo ligero:** sin `logic`, sin `past` y sin variables de sesión (−33 columnas) | 0.9144 | +0.02 | +0.00 | 16 / 14 | 9 / 8 | 0.772 | 0.62 |

- **Nada empeora el modelo.** La señal está en `future` (−4.9 sin él) y algo en `clear`. `logic`, `past` y las variables de
  sesión no aportan ni restan; quitarlas da el mismo modelo con 33 columnas menos. No sobran cosas que hagan daño.

**b) Dónde falla** (`futuro/pruebas/errores.py`, sobre `lgb_future_clear10_ses`: 1109 fallos de 13175).
- **Los fallos están en tramos enteros.** Un tramo es una racha de snapshots con la misma respuesta seguida; hay 3655 y se
  aciertan enteros el 92.7 %. **198 tramos fallados enteros dan el 82 % de los fallos**, y 75 tramos dan la mitad. El 90 %
  de los fallos está en tramos estables, no en los cambios de respuesta. Los tramos largos se aciertan peor (11–30
  snapshots: 0.80; 2–3: 0.96).
- Tramos fallados enteros por tipo: exit_coordination 45, entry_coordination 42, zone_conflict 27, return_to_route 19,
  exit_conditions 19.
- **Huella observable** de la tarea etiquetada: una instrucción compatible en ≤ 60 s, que la tarea desaparezca en ≤ 30 s o
  un cambio de su estado ligado en ≤ 60 s.

| Tramos | etiqueta con huella | top-1 del modelo con huella |
|---|---|---|
| acertados enteros | 0.877 | 0.889 |
| fallados enteros | **0.273** | 0.611 |

  - En el 39 % de los tramos fallados enteros, la etiqueta no deja huella y la top-1 del modelo sí.
  - **153 de los 198 (77 %) no tienen huella futura** y tampoco pasada: solo el 12 % tuvo una instrucción al avión en los
    60 s anteriores (29 % en los acertados sin huella futura) y el 9 % es una tarea recién aparecida.
  - Son momentos en que el experto marca una tarea con la que el controlador no hace nada visible (mirarla o pensarla): el
    modelo sigue la evidencia y la etiqueta no la tiene. Es el mismo techo que la auditoría de la sección 17.

**c) Test frente a train.** Mezcla de tipos de candidatas parecida en las 5 sesiones de test; cobertura de huella
98.5–100 % (train 98.4 %). 023 tiene listas más cortas (mediana 28–34 tareas frente a 37) y menos ritmo (`a_n300` 13–15
frente a 17), y 023sc2 tiene más candidatas de conflict_resolution (9.1 % frente a 2.3 %). Dentro de lo visto; no hay un
problema de datos en test.

**Conclusión:** la solución no está en quitar features (nada resta) ni en un desajuste de test. El ~8 % que falta son
sobre todo ~200 tramos cuya etiqueta no deja rastro observable. Se puede usar el modelo ligero, que da lo mismo con menos
columnas, pero no cambia la nota.

In [ ]:
# Reproducir (terminal, desde la raíz; 3 semillas, ~6 min cada una con --threads 5):
#   python futuro/m_lgb_future.py --sesion --threads 5 --name abl_sin_logic --feats base,past,future,clear
#   (ídem sin past / future / clear; --drop seq_len,blk_len,blk_pos,gap_prev,gap_next,x_state_age,a_n60,a_n300; ...)
#   python futuro/m_lgb_future.py --sesion --name abl_ligero --feats base,future,clear --drop seq_len,x_state_age,a_n60,a_n300
#   python futuro/pruebas/errores.py lgb_future_clear10_ses          # b) y c), sin reentrenar (~2 min)
from comparar import comparar
for c in ["abl_sin_logic_ses", "abl_sin_past_ses", "abl_sin_future_ses", "abl_sin_clear_ses", "abl_sin_vsesion_ses",
          "abl_sin_ritmo_ses", "abl_ligero_ses"]:
    comparar("lgb_future_clear_ses", c)

## 20 · Diagnósticos: qué es reducible, qué pasa entre sesiones y cómo predice cada modelo enviado (Juanma, 21/10, 02:30–03:30)

Scripts: `futuro/pruebas/f8_diag.py`, `f8_parejas.py` y `f9_explica.py` (salida en `futuro/pruebas/notas/f9_explica.out`).
Referencias: A = `suav_lgb_future_clear10_ses` (público 0.947), B = `suav_lgb_future_clear10_orcoord_ses` (público 0.934).

**20a · ¿Cuánto error es reducible?** (`f8_diag.py`, sobre A)

| Diagnóstico | Resultado | Lectura |
|---|---|---|
| Curva de aprendizaje (1.er pase, 25/50/75/100 % de los controladores de entrenamiento) | 0.870 / 0.893 / 0.898 / **0.906** | no se aplana: más controladores ayudarían; el modelo final (18 controladores) debería rendir algo por encima de la CV |
| Acuerdo entre controladores (misma tarea presente, mismo momento y escenario) | **12.7 %** | ante lo mismo, cada controlador elige distinto: es la aleatoriedad medible; el 92 % sale de las acciones del propio controlador |
| Colisiones (fila por encima con features idénticas) | 0 % de los fallos | ningún fallo es indistinguible con estos datos |
| Predecir el fallo (AUC, CV por controlador) | **0.83** | el modelo sabe cuándo duda (confianza, sin siguiente instrucción, racha larga, nº de coordinaciones) |
| Fallos con otra tarea del mismo avión por encima | **41 %** | acierta el avión, no la tarea |

**20b · Parejas del mismo avión** (`f8_parejas.py`, 14 722 parejas respuesta/hermana): el modelo pone la respuesta por encima
el 94.3 %; un clasificador dedicado a desempatar parejas (features de las dos y su diferencia), **92.1 %**: no hay más señal.
Lo que más distingue es `f_task_end`: en las tareas de acción (assume, transfer, return_route) la respuesta desaparece a ~3 s;
en coordinaciones y exit_conditions **se invierte**: desaparece a los 3 s la hermana (el transfer/assume) y la coordinación
sigue 1–3 min. Las parejas difíciles (exit_coord–return_route 84 %, exit_cond–exit_coord 84 %) no tienen instrucción propia.

**20c · Qué se transmite entre sesiones del mismo escenario**: el tráfico (una tarea aparece en el 86 % de las sesiones) y
**si una tarea se trabaja** (AUC 0.95; coordinaciones 0.83: si ninguna otra sesión la trabaja, 2 %; si más de la mitad, 70 %),
pero **no cuándo** (12.7 % de acuerdo; ~100 s de dispersión). Es lo que usa el oráculo (`or_ever`): no queda información
entre sesiones sin usar.

**20d · Cómo predice cada modelo enviado** (`f9_explica.py`, SHAP del 1.er pase, 1 semilla, 5 primeros candidatos)

| Modelo (público) | snapshot | futuro (`f_*` + `sx_*`) | pasado | otras sesiones |
|---|---|---|---|---|
| `lgb_future` (0.942) | 61 % | 33 % | 6 % | — |
| A (0.947) | 56 % | 39 % | 5 % | — |
| oráculo completo (0.940) | 44 % | 36 % | 4 % | 16 % |
| B (0.934) | 50 % | 36 % | 5 % | 9 % |

- El reparto es el mismo en CV, test 020 y test 023 (±1): los modelos no cambian de forma de predecir en test.
- Regla principal: siguiente instrucción al avión en ≤ 10 s → +1.9 al score; 10–30 s → +0.9; > 30 s → ≈ 0 (igual en CV y test).
  **El 023 tiene más snapshots sin siguiente instrucción** (38 % frente al 30 % de la CV): menos señal de futuro.
- Validación adversaria: AUC 0.999 en los dos controladores de test, por variables de tráfico de la sesión (`seq_len`,
  `a_n300`, `blk_len`); le pasaría a cualquier sesión nueva. No indica un problema.

**20e · Qué penaliza el público a B**: B saca entry_coordination del top-3 de A en el 27.5 % (020sc2) y el 27.3 % (020sc3) de
los snapshots, **por encima del percentil 90 de las sesiones de la CV** (11.6 / 17.5 / 24.9 % en p10 / p50 / p90). En la CV ese
empujón no cuesta (acierto en entry_coordination 0.748 con A y con B; B gana en exit_coord +5.5, exit_cond +2.5, zone +1.3).
**En el 023, B es típico o suave** (6.9 / 14.2 / 16.2 %). Conclusión: el 0.934 sale de un comportamiento atípico de B en las dos
sesiones del público, no de algo que se repita en el privado. Refuerza B como entrega para el 023.
Limitar el empujón del oráculo a lo visto en la CV sería ajustarse al público: no se hace.


**20f · CV a la medida del 023** (controladores de la CV más parecidos al 023 por indicadores sin etiquetas: % de top-1 de A
que es coordinación, 4.0 % en el 023 frente a 10.1 % en el 020, y snapshots sin instrucción próxima):

| Controladores | A | B (`orcoord`) | l63 | oráculo completo |
|---|---|---|---|---|
| todos (18) | 91.83 | **92.59** | 92.08 | 92.43 |
| 6 más parecidos al 023 (002, 011, 010, 006, 017, 015) | 92.25 | 92.54 | 92.49 | **92.95** |
| 3 más parecidos (002, 011, 010) | **93.03** | 92.60 | 92.29 | 92.87 |

Con controladores que coordinan poco, como el 023, **las diferencias entre modelos se reducen a ±0.4** (pocos controladores:
ruido). El oráculo gana sobre todo en coordinaciones, y el 023 coordina poco: para el privado se espera A ≈ B.
**Dirección para subir el 023:** mejorar los tipos con instrucción (zone, return_route, exit_conditions, conflict), que son
la mayoría de sus respuestas; las coordinaciones pesan poco en él.


## 21 · Bloque `instr`: qué tipo de instrucción recibe el avión y si encaja con la tarea (Juanma, 21/10, 02:00–02:45)

Script: `futuro/pruebas/f10_instr.py` (bloque `si_*`, 18 columnas; caché `futuro/pruebas/cache_instr.parquet`) y
`futuro/pruebas/f10_final.sh` (10 semillas → sesión → suavizado con CV anidada → `comparar.py` → envío).

**Por qué:** 20e–20f dicen que el 023 coordina poco y que su margen está en las tareas con instrucción; `f_next_input` /
`f_next_act` solo miran la PRIMERA instrucción siguiente. En el 16 % de las filas la instrucción compatible con la tarea no
es la primera. Columnas (sin etiquetas, de la lista de inputs de cada avión en la sesión):
`si_next_<tipo>` / `si_prev_<tipo>` (s hasta la siguiente / desde la última instrucción de cada tipo: assume, force, nivel,
lateral, transfer, otro), `si_match` / `si_prevmatch` (s hasta / desde la siguiente compatible con la tarea, tabla `compat`
de `features_future`), `si_match_rk`, `si_match_rk_flight` (rango en el snapshot y entre las tareas del avión), `si_n60`, `si_n180`.

**21a · 3 semillas + sesión** ("CV 023" = controladores 002, 011, 010, 006, 017, 015, los más parecidos al 023, sección 20f)

| Modelo | 1.er pase | + sesión | CV 023 | peor controlador |
|---|---|---|---|---|
| control `clear` (`f10_ref`) | 0.9071 | 0.9142 | 0.9186 | 0.781 |
| + `instr` | 0.9099 | 0.9153 | 0.9159 | 0.772 |
| **+ `instr` + 63 hojas + `extra_trees`** | 0.9132 | **0.9233** | **0.9256** | 0.784 |
| control con oráculo de coord. (`f10_ref_or`, = 14c) | 0.9101 | 0.9220 | 0.9219 | 0.784 |
| + `instr` + oráculo de coord. | 0.9158 | 0.9233 | 0.9243 | 0.796 |
| pseudo-etiquetas transductivas (rachas con prob. ≥ 0.97 en todos sus snapshots, 1 semilla) | — | 0.9110 (ref. 0.9132) | 0.9074 (0.9214) | 0.769 |

- `instr` solo apenas suma tras el segundo paso; **con 63 hojas + `extra_trees` sí** (+0.9 sobre el control, +0.4 sobre `l63` sin `instr`).
- Pseudo-etiquetas: los snapshots elegidos aciertan el 99.4 %, pero el modelo empeora (−0.2; −1.4 en la CV 023). Descartadas.

**21b · Comportamiento en test frente a la CV** (frente a A sin suavizar; % de snapshots de cada sesión)

| Modelo | entry_coord que salen del top-3, 020sc2 / 020sc3 | p10 / p50 / p90 en la CV | público esperado frente a A |
|---|---|---|---|
| B `orcoord` (público 0.934) | 27.1 / 26.2 | 12.5 / 18.7 / 25.6 | +1.0 (P > A 0.87) |
| `instr` + oráculo | 25.1 / 22.5 | 12.2 / 16.5 / 24.0 | +0.8 (0.83) |
| **`instr` + 63 hojas** | **10.1 / 9.3** | 6.1 / 11.1 / 19.5 | +0.8 (0.77) |

**Hipótesis H1 (el público castiga sacar entry_coordination del top-3 en el 020):** % de snapshots del 020 con entry_coordination en el
top-3 frente a la nota pública de los 4 envíos: A 60.0 % → 0.947 · `lgb_future` 54.0 % → 0.942 · oráculo completo 45.0 % → 0.940 ·
B 41.1 % → 0.934. Orden perfecto (≈ 4 % por azar con 4 puntos). El 020 coordina mucho; el 023 poco, así que pesa menos en el privado.
**H2 (el oráculo recibe valores raros en el 020): no.** `or_ever` = 0 en el 61 / 72 % de las entry_coordination del 020 en test,
igual que en la CV (64 %) y en el 023 (59–73 %). Lo distinto es que el 020 tiene muchas entry_coordination candidatas.
El modelo con `instr` + 63 hojas deja entry_coordination en el top-3 del 020 en el **67.8 %** (10 semillas).

**21c · Versión final con 10 semillas (`f10_instr_l63_s10`)**

| | 1.er pase | + sesión | + suavizado (CV anidada) | CV 023 | peor controlador |
|---|---|---|---|---|---|
| `f10_instr_l63_s10` | 0.9139 | **0.9242** | 0.9238 (−0.04: no se suaviza) | **0.9287** | 0.784 |

`comparar.py` frente a A (`suav_lgb_future_clear10_ses`): Δ **+0.59**, Δ mediana por sesión **+0.38**, sesiones 21 / 10,
controladores 13 / 4 (Δ mediana +0.69), IC 90 % **[+0.16, +1.38]**, P(media > 0) = 0.98; escenarios 0.904 / 0.934 / 0.936.

Esperado según la CV (combinando sesiones de la CV de los mismos escenarios), frente a A:

| | público (sc2 + sc3) | P(> A) | privado 023 (sc1 + sc2 + sc3) | P(> A) |
|---|---|---|---|---|
| N `f10_instr_l63_s10_ses` | +0.84 (p10 −0.7, p90 +2.2) | 0.77 | +0.58 | 0.74 |
| B `orcoord` | +0.98 (p10 −1.0, p90 +2.6) | 0.80 | +0.80 | 0.77 |

**Kaggle público: N (`sub_f10_instr_l63_s10_ses.csv`) = 0.951**, frente a 0.947 de A y 0.934 de B: el mejor envío en el público, como predecía H1.

**Conclusión:** N es el mejor modelo sin oráculo: mejora a A con todos los criterios, en la CV 023 y se comporta en test como en
la CV. **Entregas propuestas: B + N** (una con oráculo y otra sin él). Envío: `futuro/pruebas/submissions/sub_f10_instr_l63_s10_ses.csv`.


## 22 · Consejo de asesores: subir N y elegir las 2 entregas finales (Juanma, 21/10, 03:00–03:15)

Seis agentes independientes (cada uno con el mismo contexto y sin ver a los demás; el Presidente con los cinco informes).
N = `futuro/pruebas/submissions/sub_f10_instr_l63_s10_ses.csv` (fichero validado: 1863 filas, 3 `row_index` válidos y distintos).
Top-3 igual que A en el 57 % de los snapshots, que B en el 46 %, que su versión de 3 semillas en el 87 %.

| Asesor | Argumento principal | Pareja final que propone |
|---|---|---|
| Escéptico | el riesgo grave es el reglamento; mejora real de N 0 a +0.4 (selección entre ~15 variantes); peor controlador igual que A; un público alto no debe contar a favor (H1 lo infla en el 020) | N + A, o N + limpia |
| Analista crítico | maldición del ganador (Δ real ~+0.2–0.3); ruido de semillas; la calibración CV→público ya falló con B; el público no tiene potencia (±0.7) | N + A |
| Visionario | subir es barato y prueba H1 e `instr` fuera de la CV; N depende de menos reglas que B; historia para el jurado: "aprendió el flujo de trabajo del controlador, no su identidad" | N + seguridad, o mezcla N+B |
| Perfil externo | primero la elegibilidad (como en un ensayo clínico); cartera de 2 activos: diversificar; disciplina de protocolo (no decidir por la nota visible) | N + B, o N + limpia |
| Ejecutor | subir N solo como control; la regla de umbral en el público no se sostiene (error ±0.8); plan horario | según reglamento |

Correcciones de hecho del Presidente: la CV de N es 0.9242 (OOF sin suavizar; el segundo paso se ajusta por fold; el suavizado
anidado da 0.9238). El 0.9255 es la cifra optimista. `realista/` usa snapshots pasados: no es una entrega limpia.

**Resultado del envío (21/10, 03:30): N = 0.951 en el público** (no salta la alarma; por regla no cambia la decisión).

**Veredicto:**
- **Subir N** (comprobación del fichero, no criterio de decisión). Alarma solo si el público < 0.927 (A − 2.0, ~2.5 errores
  estándar): buscar fallo técnico (orden de filas, cruce de `sample_id`, comparar con la versión de 3 semillas); si no lo hay,
  A sustituye a N. Una nota alta no cuenta a favor.
- **Pareja final según el reglamento** (el riesgo de reglamento es todo o nada; el de modelo, ~1 punto en el máximo de las dos):

| El organizador dice | Marcar como las 2 entregas finales |
|---|---|
| se permiten el futuro y el oráculo | **N + B** (coinciden en el top-3 el 46 %: la pareja más diversa) |
| se permite el futuro, no el oráculo | **N + A** |
| no se permite el futuro | `submissions/sub_lgb_adri.csv` + otra limpia (`sub_nn_tf_v2.csv` o su stacking si mejora la CV) |
| **no contesta (por defecto)** | **N + `submissions/sub_lgb_adri.csv`** (siempre queda una válida) |

**Próximos pasos:** subir N y anotar la nota sin cambiar nada · escribir al organizador (¿snapshots futuros?, ¿etiquetas de otros
controladores?, ¿se revisan las dos entregas o solo la que puntúa?) · 08:00 puesta en común y preguntar en persona; hasta las 10:30
mejorar la vía limpia · 11:00 elegir con la tabla · 12:30 marcar las 2 entregas y guardar capturas · en las slides, declarar qué
información usa cada entrega.


## 23 · Exploración final, modelos teóricos, NOR y elección de las 2 entregas (Juanma, 21/10, 03:15–10:30)

Dos tandas de agentes en paralelo (cada mejora candidata con 2 verificadores adversariales: recálculo desde los OOF y búsqueda de fuga).
Scripts `futuro/pruebas/f11_*.py` (caminos nuevos) y `f12_*.py` (modelos teóricos); resultados en `resultados.jsonl` (hilos f11, f12).

**23a · Ocho caminos nuevos** (3 semillas frente a N con 3 semillas, 0.9233 con sesión, CV 023 0.9256)

| Camino | Con sesión | CV 023 | Δ | Veredicto |
|---|---|---|---|---|
| `instr` ampliado (secuencia de 3 instrucciones, acoplamiento instrucción–fin de tarea, orden entre aviones) | 0.9231 (seq) / 0.9184 (todo) | 0.9291 | −0.02 / −0.49 | ruido o peor (el orden entre aviones es estilo del controlador: no generaliza) |
| especialista en tareas con instrucción | 0.9233 (la CV anidada elige peso 0) | 0.9256 | 0 | no aporta |
| transiciones aprendidas entre tareas (forward-backward) | 0.9212 (anidada) | 0.9263 | −0.20 | no aporta |
| estilo del controlador por sesión (sin etiquetas) | 0.9229 | 0.9295 | −0.04; peor controlador −1.8 | no aporta |
| **mezcla N + oráculo** (`f11_mezclas_OR_log`, log-softmax, w = 0.5) | **0.9274 (anidada)** | **0.9317** | **+0.33** frente a N (10 sem.) | **sobrevive a 2 verificaciones**; mediana por sesión +0.00 |
| oráculo solo en exit_coordination | 0.9195 | 0.9205 | −0.38 | peor: quitar el oráculo de las entry rompe el contraste entry/exit |
| bagging por controladores (75 %) | 0.9176 | 0.9219 | −0.66 | peor |
| estimar el privado por similitud de controladores | — | — | — | no predice (falla el signo en el 020) |

**23b · Modelos teóricos** (frente a N, 10 semillas, 0.9242): Kalman / suavizador RTS 0.9219 (anidada, −0.23); riesgos competitivos
(riesgo log-lineal por tarea) 0.879 solo, cualquier peso empeora; Bayes jerárquico sobre preferencias por controlador 0.9227 (−0.15;
ajustarlo con datos sin etiquetas refuerza los propios errores); investigación operativa (asignación de episodios) 0.9241 (empate).
**Calibración (Bernoulli por snapshot + correlación en la racha + efecto aleatorio por controlador):** acierta el nivel de los 5
públicos (todos dentro del IC 90 %) pero no su orden (correlación −0.09). Privado esperado: A 0.908, B 0.912, N 0.916 (IC 90 % ±5
puntos). El efecto del controlador (±5) domina las diferencias entre modelos (±0.5): no sirve para elegir modelo.

**23c · NOR = `f10_instr_l63_or_s10_ses`** (config de N + oráculo de coordinaciones, 10 semillas): 1.er pase 0.9191, **+ sesión 0.9278**,
CV 023 0.9320, **peor controlador 0.802** (el mejor de todos). Con 3 semillas: 0.9270, peor 0.8195.
Ganancia del oráculo por tercil de coordinación del controlador (por controlador): poca +0.19 / media +0.35 / mucha +0.68 (NOR3 − N3);
B − A: +0.96 / +0.58 / +1.85. Muy variable por controlador: 002 (el más parecido al 023) NOR +1.9 y B −3.1; 006 NOR −4.0 y B +2.4.

**23d · El 023 en test:** top-1 de coordinación 3.2 % (mínimo del rango de la CV), top-1 entry_coordination 0.1 %, confianza del modelo
por debajo de la media de la CV (controlador más difícil). entry_coordination en el top-3 del 023: N 28.3 %, NOR 23.5 %, B 19.9 %
(en el 020: 67.8 / 50.8 / 41.1 %).

**23e · E[máx] por controlador:** Kaggle se queda con la mejor de las 2 entregas en el 023; se estima con la media, sobre los 18
controladores de la CV, de max(acierto X, acierto Y) (IC 90 % por bootstrap de controladores):

| Pareja | E[máx] (18) | IC 90 % | E[máx] (6 parecidos al 023) |
|---|---|---|---|
| B + NOR | **92.35** | [90.5, 94.0] | 92.20 |
| NOR + mezcla | 92.32 | [90.5, 94.0] | 92.40 |
| B + mezcla | 92.31 | | 92.32 |
| B + N | 92.17 | | 92.04 |
| **NOR + N** | 92.12 | [90.3, 93.8] | 92.34 |
| A + N | 91.90 (la peor) | | 92.29 |

**23f · Público esperado de NOR:** dos métodos que no coinciden. (1) H1 (recta de la nota pública frente al % de entry_coordination en
el top-3 del 020, 5 envíos, errores leave-one-out ≤ ±0.35): **0.941**. Hipótesis formulada a posteriori sobre 5 envíos correlacionados.
(2) CV (pares de sesiones de los escenarios 2 y 3): +0.97 sobre A → **0.957**, P(> A) 0.83; este método ya falló con B (+0.98 esperado,
−1.3 real). Rango honesto 0.935–0.960.

**23g · Decisión: NOR + N.** La CV favorece llevar oráculo (mejor media, mediana, peor controlador; información que nada más aporta), pero
los dos envíos con oráculo bajaron en el público y la CV no puede ver un fallo sistemático en test. Con una entrega con oráculo y otra sin
él, Kaggle cubre los dos casos; dos con oráculo (B + NOR) solo ganan ~0.2 en E[máx], dentro del ruido, y lo pierden todo si el oráculo
falla en el 023. Lo único bien sabido del privado es el nivel (~0.91–0.92) y que A + N es la peor pareja.


<!-- f11:s23_intro -->
## 24 · Hilo f11: mejorar N con todo lo que permiten los datos (Pablo, 21/10, 03:15–)

**Regla del organizador (confirmada por Pablo, 21/10 03:15):** se puede usar **cualquier información de los datasets**,
incluidos los snapshots futuros. El oráculo (etiquetas de otras sesiones de train/validation) entra en esa regla.

**Punto de partida:** N = `f10_instr_l63_s10` (sección 21): `clear` + `instr` + 63 hojas + `min_child` 100 + `extra_trees`,
10 semillas + segundo paso de sesión. CV **0.9242**, público **0.951**.

**Banco de pruebas** (`futuro/pruebas/f11_lab.py`, cola en paralelo `f11_cola.py`, tabla `f11_tabla.py`): cada prueba cambia
UNA cosa respecto a N y se mide con todas las métricas de la sección 14 frente a **N con las mismas semillas**
(`f11_ref_s3`, 3 semillas). Salidas en `futuro/pruebas/oof|preds/f11_*`, registro en `futuro/pruebas/resultados.jsonl`
(hilo `f11`), logs en `logs/f11_*.log`. En gpu.dsic caben 4 pruebas a la vez × 8 hilos (~3 min cada una sola, ~9 min con 4).

**Comprobación:** `f11_ref_s3` reproduce exactamente `f10_instr_l63_s3` de Juanma (1.er pase 0.9132, + sesión 0.9233,
CV 023 0.9256): el banco y las features cacheadas son las mismas.

**Solo se prueba lo que no está en las secciones 0–22.** Lo que se había probado sobre el modelo antiguo (sin `instr` ni
`extra_trees`) solo se repite si con N cambia algo esencial; por ejemplo, `extra_trees` cambia cómo influye el número de
árboles.


<!-- f11:s23_diag -->
### 24a · Dónde están los fallos de N y qué cambia el oráculo sobre N (sin reentrenar)

**Fallos de N** (`f10_instr_l63_s10_ses`; «huella» = instrucción compatible ≤ 60 s, la tarea desaparece ≤ 30 s o cambia su
estado ligado ≤ 60 s, como en la sección 19):

| Snapshots | % | acierto top-3 | fallos |
|---|---|---|---|
| la respuesta deja huella | 81.7 % | **0.978** | ~240 |
| la respuesta NO deja huella | 18.3 % (2416) | **0.685** | **761 (76 % de los fallos)** |
| algún candidato con instrucción compatible ≤ 30 s | 59.2 % | 0.953 | — |
| ninguno | 40.8 % | 0.883 | 628 |

- Sin huella, por tipo de la respuesta: entry_coordination 891 (0.79), zone_conflict 577 (0.83), exit_coordination 294
  (**0.34**), return_to_route 191 (0.57), exit_conditions 99 (0.57). En el 92 % de esos fallos el top-1 es **otro avión**.
- **El 023 tiene más snapshots sin instrucción compatible inminente:** 023sc1 44.6 %, 023sc2 54.1 %, 023sc3 64.3 % con
  alguna ≤ 30 s, frente al 59 % de la CV. 023sc1 se parece al controlador 001 (44 %), el peor de la CV. El privado
  dependerá más que la CV de los snapshots sin huella.

**Oráculo de coordinaciones sobre N, por tipo** (3 semillas + sesión, `f11_orc_s3` frente a `f11_ref_s3`, aciertos ganados):
zone_conflict **+20**, return_to_route +9, conflict_resolution +9, exit_conditions +6, non_conformance +5,
exit_coordination +3, **entry_coordination −7**. Con N, el oráculo de coordinaciones gana sobre todo **fuera** de las
coordinaciones: al bajar las coordinaciones que los demás no trabajan, suben las tareas con instrucción. Es el perfil del
023 (coordina poco; sección 20f) y explica por qué en el 020 (coordina mucho) el público lo castiga (H1, sección 21b).

### 24b · Segundo paso con varios primeros pases y con un «experto en priors» (sin reentrenar)

| Prueba | CV | Δ | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | veredicto |
|---|---|---|---|---|---|---|---|
| segundo paso conjunto: features de sesión de N **y** de N + oráculo en el mismo logit (`f11_ses2.py`) frente a `f11_orc_s3_ses` | 0.9277 | +0.09 | +0.00 | 12 / 8 | 10 / 4 | 0.781 (frente a 0.796) | ✗ ruido, baja el peor |
| segundo paso conjunto LightGBM N + XGBoost (rank:ndcg en GPU, mismas features; `f11_xgb.py`: 1.er pase 0.9042, + sesión 0.9122) frente a `f11_ref_s3_ses` | 0.9224 | −0.09 | +0.00 | 10 / 18 | 6 / 8 | 0.784 | ✗ |
| ídem con oráculo: LightGBM + XGBoost con oráculo (XGBoost: 0.9150 / 0.9235) frente a `f11_orc_s3_ses` | 0.9272 | +0.04 | +0.00 | 13 / 13 | 8 / 8 | 0.790 | ✗ |
| N + log-prob. del modelo **sin futuro** (`realista/real_todo`) con puerta «sin instrucción compatible ≤ 30 s en el snapshot» (`f11_gate.py`) frente a `f10_instr_l63_s10_ses` | 0.9252 | +0.10 | +0.00 | 14 / 10 | 10 / 6 | 0.787 | ≈ (pequeño) |

- La puerta no sirve: su peso sale −0.01. El modelo sin futuro solo entra con un peso global de 0.12 (+0.10, bootstrap
  P > 0 = 0.95, con el mismo N debajo, así que sin ruido de semillas del primer pase). **No sabe más que N justo donde N
  duda**: en los snapshots sin huella, la información que falta no está en el snapshot ni en el pasado.


<!-- f11:s23_t1 -->
### 24c · Tanda 1: información nueva y parámetros con `extra_trees` (3 semillas, frente a `f11_ref_s3` = N con las mismas semillas)

| Prueba | Qué cambia respecto a N |
|---|---|
| `pair` | conflictos (tipos 0, 1, 6): la tarea nombra dos aviones y `f_*`/`si_*` solo miran el primero; aquí, siguiente instrucción y cambio de nivel/rumbo del **segundo** avión (`f11_feats.pair`) |
| `nsi` | orden en la sesión: cuántas instrucciones (a cualquier avión) llegan antes de la siguiente a este avión (`f11_feats.nsi`) |
| `link` | diferencias entre los momentos de la evidencia de una misma fila (desaparición − instrucción compatible, respecto al final de la racha…) (`f11_feats.link`) |
| `watco05` / `watco1` | peso de cada snapshot 1/n^p según los snapshots de su controlador (todos los controladores pesan parecido) |
| `mc200`, `n800`, `n1200lr03`, `bynode06`, `cs05` | `min_child` 200; 800 árboles; 1200 árboles con lr 0.03; columnas por nodo 0.6; colsample 0.5 (con `extra_trees` no se habían probado) |

| prueba | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | CV 023 | escenarios 1/2/3 |
|---|---|---|---|---|---|---|---|---|---|
| `f11_ref_s3` | 0.9132 | **0.9233** | — | — | — | — | 0.784 | 0.9256 | 0.905 / 0.934 / 0.932 |
| `f11_pair_s3` | 0.9156 | **0.9224** | -0.08 | +0.00 | 16 / 13 | 8 / 7 | 0.793 | 0.9267 | 0.903 / 0.934 / 0.932 |
| `f11_nsi_s3` | 0.9124 | **0.9192** | -0.41 | -0.20 | 13 / 19 | 5 / 13 | 0.781 | 0.9219 | 0.901 / 0.930 / 0.927 |
| `f11_watco05_s3` | 0.9127 | **0.9224** | -0.08 | +0.00 | 13 / 17 | 7 / 11 | 0.781 | 0.9300 | 0.900 / 0.934 / 0.934 |
| `f11_link_s3` | 0.9129 | **0.9198** | -0.34 | -0.18 | 10 / 19 | 5 / 13 | 0.769 | 0.9234 | 0.900 / 0.928 / 0.933 |
| `f11_watco1_s3` | 0.9133 | **0.9218** | -0.14 | +0.00 | 15 / 16 | 7 / 10 | 0.787 | 0.9254 | 0.903 / 0.934 / 0.929 |
| `f11_mc200_s3` | 0.9142 | **0.9225** | -0.08 | +0.00 | 15 / 13 | 9 / 9 | 0.793 | 0.9247 | 0.903 / 0.932 / 0.933 |
| `f11_n800_s3` | 0.9113 | **0.9209** | -0.24 | +0.00 | 13 / 14 | 6 / 10 | 0.775 | 0.9256 | 0.899 / 0.931 / 0.934 |
| `f11_bynode06_s3` | 0.9112 | **0.9185** | -0.48 | -0.27 | 8 / 23 | 3 / 12 | 0.769 | 0.9238 | 0.898 / 0.935 / 0.923 |
| `f11_cs05_s3` | 0.9117 | **0.9203** | -0.30 | -0.24 | 11 / 19 | 5 / 12 | 0.778 | 0.9241 | 0.901 / 0.932 / 0.928 |
| `f11_n1200lr03_s3` | 0.9116 | **0.9214** | -0.18 | +0.00 | 14 / 15 | 9 / 9 | 0.787 | 0.9263 | 0.900 / 0.930 / 0.936 |

**Conclusión: nada mejora a N.** Todas las variantes quedan entre −0.08 y −0.48 con la sesión, y ninguna sube la
mediana por sesión. `pair` sube el 1.er pase (+0.24: la información del segundo avión existe), pero el segundo paso ya
lo recuperaba. Más árboles con `extra_trees` y la regularización tampoco ayudan: los parámetros de N están en su óptimo y
cualquier cambio empeora. Se confirma la lección de la sección 19: lo que falta es información, no modelo. Se cancelaron
`onehot`, `psmooth` y `l95` (parámetros) para dar paso a la información nueva de la tanda 2.


<!-- f11:s23_t2 -->
### 24d · Tanda 2: oráculo con N, oráculo sin etiquetas, objetivo listwise y segundo paso no lineal

| Prueba | Qué cambia respecto a N |
|---|---|
| `orc` | + oráculo de coordinaciones (sección 14d) sobre N: con `instr` + 63 hojas no se había medido con todas las métricas |
| `orcbase` | + oráculo completo (todas las tareas) |
| `orcgate` | + oráculo completo solo en las filas **sin huella propia** (si el controlador actúa sobre la tarea, manda su evidencia) |
| `io` | **oráculo sin etiquetas** (`f11_feats.io`): fracción de las demás sesiones del escenario (también las de test) en que el avión recibe una instrucción compatible con la tarea a ±30/60 s del mismo momento alineado, en que la tarea desaparece a ±60 s, y cuándo la resuelven. En los snapshots sin huella tiene AUC 0.59–0.64 (0.69 entre entry_coordination) |
| `ioorc` | `io` + oráculo de coordinaciones (frente a `orc`) |
| `softmax` | objetivo listwise programado a mano (entropía cruzada de la softmax en el snapshot) en vez de LambdaRank |
| `mbin63` | `max_bin` 63 (pista del Optuna de `realista/`) |

| prueba | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | CV 023 | escenarios 1/2/3 |
|---|---|---|---|---|---|---|---|---|---|
| `f11_orc_s3` | 0.9145 | **0.9268** | +0.36 | +0.50 | 22 / 10 | 13 / 5 | 0.796 | 0.9291 | 0.909 / 0.935 / 0.937 |
| `f11_io_s3` | 0.9122 | **0.9192** | -0.40 | -0.26 | 8 / 19 | 3 / 13 | 0.763 | 0.9254 | 0.901 / 0.928 / 0.930 |
| `f11_softmax_s3` | 0.9096 | **0.9175** | -0.58 | -0.44 | 7 / 21 | 4 / 13 | 0.760 | 0.9212 | 0.896 / 0.928 / 0.930 |
| `f11_ioorc_s3` | 0.9157 | **0.9243** | -0.26 | +0.00 | 14 / 18 | 6 / 11 | 0.793 | 0.9309 | 0.911 / 0.929 / 0.934 |
| `f11_mbin63_s3` | 0.9135 | **0.9239** | +0.07 | +0.00 | 17 / 14 | 9 / 7 | 0.772 | 0.9280 | 0.902 / 0.935 / 0.936 |
| `f11_orcbase_s3` | 0.9135 | **0.9221** | -0.47 | +0.00 | 11 / 18 | 6 / 11 | 0.796 | 0.9271 | 0.908 / 0.927 / 0.932 |
| `f11_orcgate_s3` | 0.9094 | **0.9173** | -0.96 | -0.55 | 8 / 25 | 3 / 13 | 0.778 | 0.9258 | 0.895 / 0.928 / 0.930 |

(`orc` frente a `f11_ref_s3`; `orcbase`, `orcgate` e `ioorc` frente a `orc`; el resto frente a `f11_ref_s3`.)

Segundo paso, sin reentrenar el primer pase:

| Prueba | CV | Δ | Δ mediana por sesión | sesiones ↑/↓ | peor controlador | veredicto |
|---|---|---|---|---|---|---|
| LightGBM pequeño (7 hojas, 100 árboles) sobre las 9 features de sesión + `task_index`, encima del logit (`f11_ses_gbm.py`), frente a `f10_instr_l63_s10_ses` | 0.9255 | +0.14 | +0.00 | 15 / 15 | 0.769 (frente a 0.784) | ✗ no robusto |

**Conclusiones:**
- **El oráculo de coordinaciones es lo único que mejora N** (+0.36; mediana por sesión +0.50; 22/10 sesiones; 13/5
  controladores; peor controlador 0.784 → 0.796; CV 023 +0.35). Cumple la regla de adopción. Con 10 semillas: sección 24e.
- El oráculo completo y el oráculo con puerta empeoran: con N, ampliar el oráculo a otras tareas resta, igual que en la
  sección 14d. Poner la puerta por huella es mucho peor (−0.96): el oráculo ayuda **también** donde el controlador actúa.
- El oráculo sin etiquetas (`io`) resta (−0.40; con oráculo, −0.26): saber cuándo actúan los demás sobre el avión no
  ayuda, y solo añade ruido de alineación. Lo que se transmite entre sesiones es **si** una tarea se trabaja (la
  etiqueta), no **cuándo** se actúa (igual que en la sección 20c).
- El objetivo listwise softmax es peor ya en el 1.er pase (0.9096 frente a 0.9132): LambdaRank con truncamiento 5 se queda.


<!-- f11:s23_orc10 -->
### 24e · N + oráculo de coordinaciones con 10 semillas (`f11_orc_s10`): mejor modelo

`python futuro/pruebas/f11_lab.py orc --orc --seeds 10 --test --threads 8 --ref f10_instr_l63_s10`

| frente a N (`f10_instr_l63_s10_ses`) | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | Δ por controlador (media / mediana) | peor controlador | CV 023 | escenarios 1/2/3 |
|---|---|---|---|---|---|---|---|---|---|---|
| N | 0.9139 | 0.9242 | — | — | — | — | — | 0.784 (001) | 0.9287 | 0.904 / 0.934 / 0.936 |
| **N + oráculo de coord.** | **0.9173** | **0.9280** | **+0.39** | **+0.38** | **22 / 11** | **15 / 2** | +0.29 / +0.50 | 0.781 (001) | **0.9304** | 0.911 / 0.936 / 0.939 |

- Cumple la regla de adopción: sube la media **y** la mediana por sesión, en el doble de sesiones de las que baja, en 15 de
  17 controladores con cambio, y el peor controlador queda igual (−0.3, ruido). Bootstrap de controladores: IC 90 %
  [−0.27, +0.69], P(media > 0) = 0.83.
- **Ganancia esperada** (sesiones de la CV combinadas como los leaderboards, `f11_privado.py`): **privado (023, escenarios
  1+2+3) +0.36** (p10 −0.92, p90 +1.34; P(> N) = 0.71); público (020, escenarios 2+3) +0.21 (p10 −1.57, p90 +1.23; 0.68).
- **En test** (`f11_check.py`): mismo top-3 que N en el 55–70 % de los snapshots. Saca entry_coordination del top-3: en el
  020 pasa de 55/53 % a 42/42 % (020sc2/020sc3) y en el 023 de 29/24/27 % a 21/21/23 %. Es el patrón de H1 (sección 21b):
  **en el público puede quedar por debajo de N** aunque la CV diga lo contrario. Por la regla del equipo (se decide por la
  CV, y el privado es el 023, que coordina poco) eso no cambia la decisión.
- Envío: **`futuro/pruebas/submissions/sub_f11_orc_s10_ses.csv`** (validado: 1863 snapshots, 3 `row_index` distintos y
  existentes). Sin suavizado (en N restaba; se mide en 24f).


<!-- f11:s23_mix -->
### 24f · El oráculo veta tareas: mezcla aritmética de C y N (`f11_mezcla.py`, sin reentrenar)

**Problema** (C frente a N por controlador): C mejora a 15 de 17 controladores, pero el **006 cae −4.5** (0.917 → 0.872).
En 006sc2, N acierta 18 de los 21 snapshots cuya respuesta es entry_coordination y C ninguno: es una coordinación que
casi ningún otro controlador hace en ese escenario, y el oráculo la hunde aunque la evidencia propia sea fuerte. El 006
está entre los 6 controladores más parecidos al 023 (sección 20f). El segundo paso y los stacks combinan en log (producto
de expertos), así que un ~0 del oráculo veta la tarea.

**Mezcla aritmética de probabilidades:** p = (1 − w) · p_C + w · p_N (softmax de cada modelo ya con segundo paso). Un
modelo solo ya no puede vetar. Frente a N (`f10_instr_l63_s10_ses`); «esperado» = sesiones de la CV combinadas como en
el privado (escenarios 1+2+3) o el público (2+3) (`f11_privado.py`):

| w (peso de N) | CV | Δ | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | 006 | privado esperado (p10, p90; P > N) | público esperado (p10; P > N) |
|---|---|---|---|---|---|---|---|---|---|
| 0 (C) | 0.9280 | +0.39 | +0.38 | 22 / 11 | 15 / 2 | 0.781 | 0.872 | +0.36 (−0.92, +1.34; 0.71) | +0.21 (−1.57; 0.68) |
| 0.2 | 0.9283 | +0.42 | +0.30 | 21 / 7 | 15 / 2 | 0.787 | 0.888 | +0.39 (−0.58, +1.22; 0.74) | +0.22 (−1.10; 0.69) |
| 0.3 | 0.9285 | +0.43 | +0.29 | 22 / 7 | 14 / 2 | 0.787 | 0.896 | +0.42 (−0.24, +1.20; 0.77) | +0.30 (−0.60; 0.72) |
| 0.4 | 0.9284 | +0.43 | +0.27 | 21 / 7 | 13 / 3 | 0.781 | 0.912 | +0.42 (+0.00, +1.00; 0.82) | +0.35 (−0.13; 0.79) |
| **0.5 (M50)** | 0.9281 | +0.39 | +0.27 | 22 / 7 | 13 / 3 | 0.778 | 0.920 | **+0.40 (−0.08, +0.89; 0.82)** | **+0.36 (+0.00; 0.82)** |

- La media no cambia con w (CV anidada eligiendo w: 0.9276; curva plana entre 0.05 y 0.5). **Lo que cambia es el riesgo:**
  frente a N bajan 7 sesiones en lugar de 11, la cola mala del privado esperado pasa de −0.92 a ~0, y el 006 se recupera
  (0.920). Para un privado de un solo controlador eso vale más que la media.
- Se elige **w = 0.5 (M50, media simple)**: no se ajusta ningún peso y queda arriba en las dos simulaciones.
  Suavizado temporal sobre C (CV anidada): 0.9280 → 0.9281, no se usa.

**Mejor de 2 entregas** (Kaggle puntúa la mejor de las 2 elegidas). E[máx] simulado en un controlador nuevo como el 023:

| pareja | C + M50 | M40 + B | C + B | M50 + B | C + M40 | N + C | M40 + A |
|---|---|---|---|---|---|---|---|
| E[máx] (privado simulado) | **93.07** | 93.06 | 93.05 | 93.05 | 93.04 | 92.98 | 92.92 |

Medias sueltas en el privado simulado: M40 92.85 · M50 92.83 · C 92.79 · B 92.61 · N 92.43 · A 91.79. En el público
simulado (020) la mejor pareja es M50 + B (93.99). **Las parejas buenas llevan una entrega con oráculo; entre ellas las
diferencias (< 0.1) son ruido.**


<!-- f11:s23_final -->
### 24g · Tanda 3 y versiones finales con 20 semillas

**Tanda 3** (3 semillas; `pairorc` y `causalorc` frente a `orc`, el resto frente a `f11_ref_s3`):

| Prueba | Qué cambia |
|---|---|
| `pairorc` | segundo avión de los conflictos + oráculo de coordinaciones |
| `causal` / `causalorc` | bloque `cz_*` de `realista/` (segundos desde el ÚLTIMO cambio del estado de la tarea), sin / con oráculo |
| `dart` | DART (drop 0.1, skip 0.5) con `extra_trees` |
| `mono` | restricción monótona: antes la instrucción compatible (`si_match`, `si_match_rk`) → más probable |
| `n300` / `n250` | menos árboles (800 restaba) |

| prueba | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | CV 023 | escenarios 1/2/3 |
|---|---|---|---|---|---|---|---|---|---|
| `f11_pairorc_s3` | 0.9166 | **0.9249** | -0.19 | +0.00 | 11 / 18 | 6 / 11 | 0.784 | 0.9280 | 0.908 / 0.938 / 0.928 |
| `f11_causal_s3` | 0.9117 | **0.9191** | -0.42 | +0.00 | 11 / 17 | 7 / 9 | 0.781 | 0.9183 | 0.897 / 0.932 / 0.929 |
| `f11_dart_s3` | 0.9091 | **0.9129** | -1.03 | -0.79 | 7 / 23 | 3 / 14 | 0.760 | 0.9221 | 0.892 / 0.929 / 0.918 |
| `f11_mono_s3` | 0.9135 | **0.9208** | -0.25 | +0.00 | 12 / 16 | 9 / 9 | 0.790 | 0.9234 | 0.901 / 0.931 / 0.931 |
| `f11_causalorc_s3` | 0.9173 | **0.9274** | +0.05 | +0.00 | 14 / 14 | 10 / 7 | 0.808 | 0.9273 | 0.909 / 0.935 / 0.939 |
| `f11_n300_s3` | 0.9134 | **0.9220** | -0.12 | +0.00 | 9 / 14 | 6 / 8 | 0.781 | 0.9225 | 0.906 / 0.933 / 0.928 |
| `f11_n250_s3` | 0.9148 | **0.9213** | -0.20 | +0.00 | 7 / 14 | 5 / 8 | 0.790 | 0.9247 | 0.905 / 0.932 / 0.927 |

Nada mejora: `causalorc` empata con `orc` (+0.05, 14/14 sesiones, mediana igual). Con esto quedan probados en la sección 24
10 cambios de features (5 bloques nuevos, 2 variantes del oráculo y 3 combinaciones), 14 de parámetros, objetivo o modelo (incluido XGBoost en GPU) y 8 segundos pasos, mezclas o posprocesos: **solo el oráculo de coordinaciones mejora N**, y la mezcla aritmética con N le quita riesgo (24f).

**Versiones finales con 20 semillas** (semillas 0–9 + 10–19, media del 1.er pase y segundo paso encima, `f11_promedio.py`):

| Modelo | CV | Δ frente a N (10 s.) | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | CV 023 | 006 | envío |
|---|---|---|---|---|---|---|---|---|---|
| N20 (sin oráculo) | 0.9249 | +0.07 | +0.00 | 15 / 6 | 10 / 5 | 0.784 | 0.9282 | 0.920 | `sub_f11_N20_ses.csv` |
| **C20** (N + oráculo de coord.) | **0.9285** | **+0.43** | **+0.55** | 21 / 9 | 14 / 2 | **0.793** | 0.9315 | 0.880 | `sub_f11_C20_ses.csv` |
| M50 (0.5·C20 + 0.5·N20, probabilidades) | 0.9278 | +0.36 | +0.25 | 19 / 8 | 11 / 4 | 0.778 | **0.9328** | 0.914 | `sub_f11_M50_20.csv` |

Esperado frente a N20 (sesiones de la CV combinadas como el leaderboard):

| | privado (023): Δ (p10, p90) · P(> N20) | público (020): Δ (p10, p90) · P(> N20) |
|---|---|---|
| C20 | +0.35 (−1.02, +1.34) · 0.71 | +0.23 (−1.65, +1.50) · 0.69 |
| M50 | +0.30 (−0.19, +1.00) · 0.69 | +0.32 (−0.27, +0.85) · 0.69 |

**Mejor de 2 entregas** (E[máx] simulado), privado: C20 + B 93.06 · C20 + M50 93.04 · **N20 + C20 93.03** · M50 + B 93.00 ·
N20 + B 92.85 · N20 + M50 92.83. Público: M50 + B 93.93 · C20 + B 93.93 · N20 + C20 93.88.

**Recomendación (información permitida = todos los datasets): C20 + N20.** Las parejas de arriba empatan (< 0.05). Esta es
la más simple y la más diversa (una entrega con oráculo y otra sin él; mismo top-3 en solo el 57–70 % de los snapshots de
test), y cubre el riesgo de que el oráculo falle en el 023 como en el 006. N20 es N (público 0.951) con 10 semillas más
(mismo top-3 que N en el 91–96 % de los snapshots). Todos validados con `f11_check.py`.


<!-- f11:s23_m30 -->
### 24h · M30: mezcla 0.7·C20 + 0.3·N20 elegida en CV anidada (mejor modelo) y pareja final

Con los modelos de 20 semillas, la CV anidada de la mezcla aritmética (también con un peso distinto para entry_coordination)
elige **el mismo peso uniforme w = 0.3 en los 5 folds** → CV anidada **0.9293**. Dar más peso a N en entry_coordination
resta (w_entry 0.5–1.0: 0.9252–0.9272), así que las pérdidas de C en las entradas se compensan en el resto.

| Modelo | CV | Δ frente a N20 | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | CV 023 | 006 | privado esperado frente a N20 (p10, p90; P) | público esperado (p10; P) |
|---|---|---|---|---|---|---|---|---|---|---|
| N20 | 0.9249 | — | — | — | — | 0.784 | 0.9282 | 0.920 | — | — |
| C20 | 0.9285 | +0.36 | +0.42 | 22 / 9 | 14 / 3 | 0.793 | 0.9315 | 0.880 | +0.35 (−1.02, +1.34; 0.71) | +0.23 (−1.65; 0.69) |
| **M30** `sub_f11_M30_20.csv` | **0.9293** | **+0.45** | **+0.40** | **21 / 8** | **14 / 3** | 0.790 | **0.9342** | 0.909 | **+0.45 (−0.20, +1.22; 0.78)** | **+0.41 (−0.41; 0.74)** |
| M50 | 0.9278 | +0.30 | +0.00 | 17 / 9 | 11 / 4 | 0.778 | 0.9328 | 0.914 | +0.30 (−0.19, +1.00; 0.69) | +0.32 (−0.27; 0.69) |

Frente a C20, M30 da +0.08 con la mediana igual (12/10 sesiones): no sube más la media que C20; lo que mejora es el riesgo.

**Mejor de 2 entregas** (E[máx] simulado en un controlador nuevo; p10 entre paréntesis):

| pareja | privado (023) | público (020) |
|---|---|---|
| **M30 + B** (`sub_suav_lgb_future_clear10_orcoord_ses`) | **93.10** (89.40) | **93.99** (90.65) |
| M30 + C20 | 93.05 (89.38) | 93.84 (90.37) |
| C20 + N20 | 93.03 (89.29) | 93.88 (90.37) |
| M30 + N20 | 92.99 (89.34) | 93.82 (89.67) |
| N20 + A (sin oráculo) | 92.63 (88.75) | 93.47 (89.18) |

~~**Recomendación: M30 + B.**~~ (revisada en 24l: M30 + N) Es la mejor pareja en los dos leaderboards simulados; B aporta diversidad (otro primer pase,
con suavizado). Si se quiere una entrega sin oráculo como cobertura: **M30 + N20** (−0.1). Las diferencias entre las
parejas con oráculo (< 0.1) son ruido; frente a la pareja sin oráculo, todas ganan unas 0.4.


<!-- f11:s23_i -->
### 24i · Más variantes del oráculo y de la mezcla (sin mejora)

| prueba | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | CV 023 | escenarios 1/2/3 |
|---|---|---|---|---|---|---|---|---|---|
| `f11_orcBc_s3` | 0.9169 | **0.9271** | +0.02 | +0.00 | 16 / 15 | 8 / 7 | 0.772 | 0.9282 | 0.909 / 0.933 / 0.940 |
| `f11_orcEc_s3` | 0.9166 | **0.9264** | -0.05 | +0.00 | 15 / 14 | 9 / 8 | 0.749 | 0.9335 | 0.902 / 0.939 / 0.939 |
| `f11_orcexit_s3` | 0.9117 | **0.9210** | -0.58 | -0.57 | 8 / 25 | 4 / 14 | 0.775 | 0.9260 | 0.899 / 0.936 / 0.929 |

(frente a `orc`; `orcBc`/`orcEc` = variantes B/E del oráculo de la sección 12b, que con N no se habían medido, ahora solo
en coordinaciones; `orcexit` = oráculo solo en exit_coordination.)

- Las variantes ricas empatan con el oráculo de coordinaciones de siempre (+0.02 / −0.05) y bajan al peor controlador.
- **Quitar el oráculo de entry_coordination resta mucho (−0.58)**: aunque veta alguna coordinación rara (006), en media
  las entradas son donde más aporta. Por eso la mezcla con N (24f, 24h) es mejor solución que recortar el oráculo.
- Mezcla de tres (C20 + N20 + B, pesos en CV anidada): 0.9292, igual que M30 (0.9293). B no aporta dentro de la mezcla;
  sirve más como segunda entrega por diversidad (24h).
- Oráculo solo en el segundo paso (prior en log con pesos aprendidos, `f11_ses_orc.py`, primer pase N20): 0.9257 (+0.08
  frente a N20, −0.28 frente a C20). El oráculo rinde dentro del primer pase, cruzado con la evidencia propia.


<!-- f11:s23_j -->
### 24j · Tanda 4 (sin mejora) y versiones finales con 30 semillas

**Tanda 4:**

| prueba | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | CV 023 | escenarios 1/2/3 |
|---|---|---|---|---|---|---|---|---|---|
| `f11_gr03_s3` | 0.8430 | **0.9066** | -1.67 | -1.55 | 8 / 28 | 3 / 13 | 0.751 | 0.9104 | 0.884 / 0.923 / 0.914 |
| `f11_gr03orc_s3` | 0.8417 | **0.9010** | -2.58 | -2.57 | 5 / 32 | 1 / 17 | 0.734 | 0.9047 | 0.873 / 0.925 / 0.905 |
| `f11_cfl_s3` | 0.9105 | **0.9195** | -0.37 | -0.26 | 14 / 20 | 7 / 11 | 0.775 | 0.9238 | 0.900 / 0.932 / 0.928 |
| `f11_cflorc_s3` | 0.9173 | **0.9272** | +0.04 | +0.00 | 13 / 15 | 8 / 8 | 0.772 | 0.9320 | 0.908 / 0.939 / 0.935 |

- `gr03` / `gr03orc`: **relevancia graduada con varios anotadores**: respuesta propia = 2 y tarea que otros controladores
  trabajan a ±60 s (or_near60 del oráculo completo ≥ 0.3, 2.6 tareas por snapshot) = 1. **Hunde el primer pase** (0.913 →
  0.843): las tareas «plausibles» se cuelan por encima de la respuesta (−1.7 / −2.6 con sesión). Se cancelaron las de 0.5.
- `cfl` / `cflorc`: QUÉ hace el siguiente cambio de nivel autorizado (¿al nivel de salida?, ¿lo acerca?, salto en FL).
  Para exit_conditions la señal existe (nivel de salida en ≤ 60 s: 11.6 % de las respuestas frente al 0.4 % del resto),
  pero es rara y el modelo no gana (−0.37 / +0.04).
- Mezclar los PRIMEROS pases de C20 y N20 (w = 0.3) y aplicar el segundo paso encima: 0.9296, igual que M30 (+0.02, 6/4
  sesiones). Se queda la mezcla al final.

**Versiones finales con 30 semillas** (tres grupos de 10: 0–9, 10–19, 20–29). Ruido entre grupos de 10 semillas: N da
0.9242 / 0.9236 / 0.9211 y C 0.9280 / 0.9290 / 0.9286, así que **±0.15 entre versiones es ruido de semillas** incluso
con 10. Con 30, la CV no sube, pero las predicciones de test son más estables.

| Modelo | CV | Δ frente a N20 | mediana por sesión | peor controlador | CV 023 | privado esperado frente a N30 (p10; P) | envío |
|---|---|---|---|---|---|---|---|
| N30 | 0.9234 | −0.14 | 0.929 | 0.781 | 0.9273 | — | `sub_f11_N30_ses.csv` |
| C30 | 0.9283 | +0.35 | 0.935 | 0.793 | 0.9320 | +0.50 (−0.73; 0.75) | `sub_f11_C30_ses.csv` |
| **M30_30** = 0.7·C30 + 0.3·N30 | **0.9290** | **+0.41** | 0.931 | **0.793** | **0.9344** | **+0.56 (−0.17; 0.81)** | **`sub_f11_M30_30.csv`** |

(CV anidada del peso con 30 semillas: 0.9282, w = 0.1–0.3 según el fold; se mantiene w = 0.3, el elegido con 20.)

**Mejor de 2 entregas con las versiones de 30 semillas** (E[máx] simulado; p10):

| pareja | privado (023) | público (020) |
|---|---|---|
| **M30_30 + B** | **93.08** (89.41) | **93.99** (90.35) |
| C30 + B | 93.04 (89.34) | 93.95 (90.66) |
| C30 + M30_30 | 93.02 (89.42) | 93.85 (90.37) |
| N30 + C30 | 92.97 (89.34) | 93.83 (90.37) |
| M30_30 + N (público 0.951) | 92.92 (89.38) | 93.75 (89.84) |
| N + A (las dos sin oráculo) | ~92.6 | ~93.5 |

~~Recomendación final~~ (revisada en 24l: **M30_30 + N**) **(información permitida = todos los datasets): `futuro/pruebas/submissions/sub_f11_M30_30.csv` +
`futuro/submissions/sub_suav_lgb_future_clear10_orcoord_ses.csv` (B).** Si se quiere una entrega sin oráculo como
cobertura: M30_30 + N (`sub_f10_instr_l63_s10_ses.csv`, público 0.951), −0.16. Todos los envíos están validados.


<!-- f11:s23_k -->
### 24k · Tanda 5: el objetivo LambdaRank (truncamiento, normalización, sigmoid) y pesos con oráculo

| prueba | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | CV 023 | escenarios 1/2/3 |
|---|---|---|---|---|---|---|---|---|---|
| `f11_trunc3_s3` | 0.9126 | **0.9194** | -0.39 | -0.26 | 12 / 22 | 5 / 10 | 0.772 | 0.9236 | 0.897 / 0.933 / 0.928 |
| `f11_nonorm_s3` | 0.9156 | **0.9225** | -0.08 | +0.00 | 13 / 14 | 9 / 8 | 0.784 | 0.9280 | 0.904 / 0.932 / 0.933 |
| `f11_trunc3orc_s3` | 0.9172 | **0.9304** | +0.36 | +0.41 | 19 / 12 | 10 / 5 | 0.802 | 0.9309 | 0.915 / 0.938 / 0.939 |
| `f11_nonormorc_s3` | 0.9174 | **0.9264** | -0.05 | +0.00 | 14 / 16 | 7 / 9 | 0.784 | 0.9300 | 0.906 / 0.936 / 0.938 |
| `f11_trunc4orc_s3` | 0.9176 | **0.9271** | +0.02 | +0.00 | 12 / 15 | 8 / 9 | 0.778 | 0.9304 | 0.909 / 0.933 / 0.941 |
| `f11_trunc2orc_s3` | 0.9180 | **0.9277** | +0.09 | +0.00 | 16 / 12 | 9 / 7 | 0.769 | 0.9322 | 0.907 / 0.938 / 0.940 |
| `f11_sig2orc_s3` | 0.9186 | **0.9268** | +0.00 | +0.00 | 16 / 15 | 7 / 9 | 0.805 | 0.9295 | 0.914 / 0.934 / 0.933 |
| `f11_sig05orc_s3` | 0.9146 | **0.9236** | -0.32 | +0.00 | 9 / 16 | 4 / 12 | 0.793 | 0.9295 | 0.903 / 0.933 / 0.937 |
| `f11_watcoorc_s3` | 0.9173 | **0.9277** | +0.09 | +0.00 | 13 / 15 | 6 / 10 | 0.796 | 0.9346 | 0.910 / 0.940 / 0.935 |
| `f11_t3n10_s10` | 0.9135 | **0.9221** | -0.20 | +0.00 | 11 / 14 | 6 / 8 | 0.784 | 0.9276 | 0.903 / 0.936 / 0.928 |
| `f11_t3orc10_s10` | 0.9189 | **0.9295** | +0.14 | +0.00 | 12 / 12 | 8 / 8 | 0.802 | 0.9339 | 0.913 / 0.936 / 0.940 |

(sin oráculo frente a `f11_ref_s3`; con oráculo frente a `f11_orc_s3`; `t3orc10` frente a `f11_orc_s10` y `t3n10` frente a N.)

- **Truncamiento 3 con oráculo**: +0.36 con 3 semillas, pero la curva 2 / 3 / 4 / 5 = +0.09 / +0.36 / +0.02 / 0 no es
  suave, y **con 10 semillas se queda en +0.14** (mediana igual, 12/12 sesiones): ruido de semillas. Sin oráculo resta
  (−0.39 con 3 semillas, −0.20 con 10). Se queda el truncamiento 5.
- `lambdarank_norm` desactivado, `sigmoid` 0.5 / 2 y peso por controlador con oráculo: ruido o peor.
- **Lección de método:** con 3 semillas solo son fiables efectos > ~0.3, y aun así hay que confirmarlos con 10 (aquí,
  +0.36 → +0.14). Entre grupos de 10 semillas del mismo modelo, la CV varía ±0.15 (24j).


<!-- f11:s23_l -->
### 24l · Qué esperar en el público con M30 (H1) y pareja final

**H1 con los 5 envíos que tienen nota** (% de snapshots del 020 con alguna entry_coordination en el top-3, `f11_check.py`):

| envío | entry_coordination en top-3 (020) | público |
|---|---|---|
| N | 54.0 % | 0.951 |
| A | 47.7 % | 0.947 |
| lgb_future | 44.0 % | 0.942 |
| oráculo completo | 36.7 % | 0.940 |
| B | 33.6 % | 0.934 |

Ajuste: público ≈ 0.910 + 0.077 · x (r = 0.97; 5 puntos). **Predicho:** M30_30 (45.4 %) → **0.944**; C30 (42.6 %) → 0.942;
N30 (54.0 %) → 0.951.

- Si M30 sale ~0.944 en el público, **es lo esperado y no indica un fallo técnico**. Por la regla del equipo se decide por la CV.
- Pero también es una prueba real (5 de 5 envíos) de que en un controlador nuevo **la CV puede sobrestimar el oráculo**:
  para el 020 la CV esperaba ganancia (020sc1 de validación ganaba) y el público dice lo contrario. El 023 coordina poco
  (top-1 coordinación 3–5 % frente al 9–11 % del 020, sección 14f), así que debería afectarle menos, pero no hay garantía.

**Pareja final recomendada: M30_30 + N** (`futuro/pruebas/submissions/sub_f11_M30_30.csv` +
`futuro/pruebas/submissions/sub_f10_instr_l63_s10_ses.csv`, público 0.951, ya subido). Una entrega con oráculo y otra sin
él: si el oráculo transfiere al 023 como dice la CV, cuenta M30 (privado esperado +0.56 frente a N30); si falla como en el
020, cuenta N. Cuesta −0.16 de E[máx] simulado frente a M30 + B (93.08), pero cubre el fallo que ya se ha visto.
**M30 + B** (las dos con oráculo) solo si se confía del todo en la CV.


<!-- f11:s23_m -->
### 24m · Confirmaciones con 10 semillas, `coord` sobre N y oráculo también en el segundo paso (sin mejora)

| prueba | 1.er pase | + sesión | Δ | Δ mediana por sesión | sesiones ↑/↓ | controladores ↑/↓ | peor controlador | CV 023 | escenarios 1/2/3 |
|---|---|---|---|---|---|---|---|---|---|
| `f11_coordorc_s3` | 0.9155 | **0.9243** | -0.26 | +0.00 | 14 / 18 | 7 / 11 | 0.781 | 0.9278 | 0.904 / 0.934 / 0.936 |
| `f11_coordN_s3` | 0.9123 | **0.9180** | -0.52 | -0.30 | 8 / 24 | 3 / 15 | 0.781 | 0.9197 | 0.901 / 0.927 / 0.927 |
| `f11_mbin63orc10_s10` | 0.9179 | **0.9275** | -0.05 | +0.00 | 12 / 15 | 7 / 10 | 0.805 | 0.9295 | 0.910 / 0.935 / 0.939 |
| `f11_watcoorc10_s10` | 0.9173 | **0.9280** | -0.01 | +0.00 | 12 / 15 | 6 / 12 | 0.808 | 0.9324 | 0.908 / 0.939 / 0.937 |
| `f11_causalorc10_s10` | 0.9154 | **0.9278** | -0.02 | +0.00 | 11 / 12 | 7 / 8 | 0.805 | 0.9265 | 0.909 / 0.936 / 0.941 |

(los `*10` frente a `f11_orc_s10`; `coordorc` frente a `f11_orc_s3`; `coordN` frente a `f11_ref_s3`.)

- Lo que con 3 semillas parecía algo positivo con oráculo (`causal` +0.05 y peor controlador 0.808, peso por controlador
  +0.09, `max_bin` 63) da **0 con 10 semillas** (−0.02 / −0.01 / −0.05).
- El bloque `coord` (`sc_*`), recomprobado sobre N porque `instr` solo funcionó con 63 hojas + `extra_trees`, sigue
  restando: −0.26 con oráculo, −0.53 sin él.
- Oráculo **también** en el segundo paso (prior en log, `f11_ses_orc.py`, encima de C30; la variante F de 12c): C30 0.9283
  → 0.9299 (+0.16, 16/11 sesiones, mediana igual, P > 0 = 0.86), pero dentro de la mezcla con N30 se solapa con lo que ya
  da la mezcla: 0.9293 frente a 0.9290 de M30_30 (+0.04, 14/13). No compensa la complejidad.
- Promedio de C30 con el truncamiento 3 (`t3orc10`): +0.05 en C y +0.02 en la mezcla. Nada.


<!-- f11:s23_n -->
### 24n · Bagging con parámetros sacudidos y 40 semillas (sin mejora) · cierre de la sección 24

`--jitter`: cada semilla toma hojas ∈ {47, 63, 79}, `min_child` ∈ {80, 100, 120}, colsample ∈ {0.6, 0.7, 0.8} y
subsample ∈ {0.7, 0.8, 0.9} al azar (semillas 30–39), para que los modelos promediados sean más distintos que con solo la semilla.

| | CV | frente a |
|---|---|---|
| C con parámetros sacudidos (10 semillas) | 0.9283 | +0.02 frente a C10 (`f11_orc_s10`) |
| N con parámetros sacudidos (10 semillas) | 0.9226 | −0.16 frente a N (dentro del ±0.15 entre grupos) |
| C40 = C30 + sacudidas | 0.9283 | +0.00 frente a C30 |
| N40 = N30 + sacudidas | 0.9232 | −0.02 frente a N30 |
| M30 con C40 y N40 | 0.9283 | −0.07 frente a M30_30 (4/7 sesiones) |

Las semillas están saturadas desde ~20: **M30_30 se queda como versión final.**

**Resumen de la sección 24** (≈ 50 pruebas, todas con las métricas de la sección 14):
- **Mejora:** el oráculo de coordinaciones sobre N (C: +0.4) y la mezcla aritmética con N, que le quita el veto a las
  coordinaciones raras (M30: 0.9290 con 30 semillas, frente a 0.9242 de N; CV 023 0.9344; privado esperado +0.56 frente a N30).
- **No mejora (no repetir):** parámetros de LightGBM alrededor de N (árboles, hojas, `min_child`, colsample, por nodo,
  `max_bin`, DART, truncamiento, normalización, sigmoid, monótonas, parámetros sacudidos), objetivo softmax, XGBoost en GPU,
  pesos por controlador, bloques nuevos (segundo avión de los conflictos, orden en la sesión, diferencias de evidencia,
  oráculo sin etiquetas, `cz_*`, qué hace el siguiente nivel autorizado, `coord`), variantes del oráculo (completo, con
  puerta, B/E en coordinaciones, solo salidas), relevancia graduada con otros controladores, segundos pasos nuevos
  (conjunto, no lineal, con experto sin futuro, con prior del oráculo) y más semillas.
- **Entrega:** `sub_f11_M30_30.csv` + `sub_f10_instr_l63_s10_ses.csv` (N), sección 24l.


## Historial completo

In [ ]:
historial()